# Constants and Configs

In [1]:
import datetime as dt

# Data parameters
START_DATE = dt.datetime(2006, 11, 1)
END_DATE = dt.datetime(2025, 11, 12)
TICKER = 'SPY'  # S&P 500

# Portfolio parameters
INITIAL_CAPITAL = 100_000
COMMISSION_PER_TRADE = 0

# Feature normalization parameters
ZSCORE_WINDOW = 200  # Window for z-score normalization of features (price, coincident signals, etc.)

# Coincident index windows for multi-timeframe analysis
COINCIDENT_WINDOWS = [5, 20, 252]  # Short, medium, and long-term momentum windows
MULTI_COINCIDENT_WINDOW = 20  # Window for multi-coincident index strategies

# Strategy parameters
MOMENTUM_PARAMS = {
    'sma_short_long_pairs': [(20, 60), (50, 200), (24, 58)],
    'macd_params': [
        (12, 26, 9),   # Standard MACD
        (5, 35, 5),    # Faster MACD
        (19, 39, 9),   # Slower MACD
    ],
}

MEAN_REVERSION_PARAMS = {
    'rsi_period': [14, 21],
    'rsi_threshhold_pairs': [(30, 70), (25, 75)],
    'rsi_overbought': 70,
    'rsi_oversold': 30,
    'zscore_window': 42,
    'zscore_threshold': 2,
    'bollinger_windows': [20, 30, 50],
    'bollinger_std_devs': [1.5, 2.0, 2.5],
}

# Volume and oscillator strategy parameters
VOLUME_OSCILLATOR_PARAMS = {
    'enabled': True,
    'obv': [
        ('SPY', 20),   # ticker, window
        ('SPY', 50),
    ],
    'stochastic': [
        ('SPY', 14, 3, 20, 80),  # ticker, k_period, d_period, oversold, overbought
        ('SPY', 5, 3, 20, 80),   # Faster stochastic
    ],
    'roc': [
        (12, 0),   # period, threshold
        (20, 0),
        (5, 0),    # Faster ROC
    ],
}

ML_PARAMS = {
    'train_test_split': 0.75,
    'random_seed': 1111,
    'models': ['RandomForest', 'GradientBoosting', 'LASSO'],
    'tune_hyperparameters': True,  # Use GridSearchCV to tune each model
    'n_jobs': 1,  # Number to 1 to manage resource usage
    'ic_percentile': 20,  # Top percentile of features by Information Coefficient (0-100). None = no filtering
    'correlation_threshold': 1,  # Threshold for removing highly correlated features in Stage 2 filtering (0-1)
    'nan_threshold': 0.3,  # Threshold for removing columns with high NaN percentage (0-1). E.g., 0.3 = remove columns with >30% NaNs
    # Debug/controls for feature filtering
    'feature_filter_debug': True,  # Prints detailed keep/remove decisions
    # Optionally force-keep certain features regardless of correlation filtering (full column names)
    # e.g., ['Coincident_TLT_5__TLT_signal']
    'force_keep_features': ['Coincident_TLT_5__TLT_signal'],
}

# Common coincident indices for strategy use
CURRENCY_INDICES = {
    'DXY': 'DX-Y.NYB',  # US Dollar Index
    'EUR': 'EURUSD=X',   # Euro/USD
    'JPY': 'JPY=X',      # Japanese Yen
    'GBP': 'GBPUSD=X',   # British Pound
}

MARKET_INDICES = {
    'VIX': '^VIX',       # Volatility Index
    'TNX': '^TNX',       # 10-Year Treasury Yield
    'GLD': 'GLD',        # Gold ETF
    'TLT': 'TLT',        # 20+ Year Treasury Bond ETF
    'USO': 'USO',        # Oil ETF
    'UUP': 'UUP',        # US Dollar Bullish ETF
    # SPY Industry/Sector ETFs
    'XLK': 'XLK',        # Technology Select Sector SPDR
    'XLF': 'XLF',        # Financial Select Sector SPDR
    'XLV': 'XLV',        # Health Care Select Sector SPDR
    'XLE': 'XLE',        # Energy Select Sector SPDR
    'XLI': 'XLI',        # Industrial Select Sector SPDR
    'XLP': 'XLP',        # Consumer Staples Select Sector SPDR
    'XLY': 'XLY',        # Consumer Discretionary Select Sector SPDR
    'XLU': 'XLU',        # Utilities Select Sector SPDR
    'XLRE': 'XLRE',      # Real Estate Select Sector SPDR
    'XLB': 'XLB',        # Materials Select Sector SPDR
    'XLC': 'XLC',        # Communication Services Select Sector SPDR
}

# Coincident indices strategy parameters
COINCIDENT_INDICES_PARAMS = {
    'enabled': True,
    'single_indices': [
        # SPY itself (rolling mean of returns as momentum signal) - multiple windows
        *[('SPY', window) for window in COINCIDENT_WINDOWS],
        # All currency indices with multiple windows
        *[(ticker, window) for ticker in CURRENCY_INDICES.values() for window in COINCIDENT_WINDOWS],
        # All market indices with multiple windows
        *[(ticker, window) for ticker in MARKET_INDICES.values() for window in COINCIDENT_WINDOWS],
    ],
    'multi_indices': [
        # All currency indices combined with mean aggregation
        (list(CURRENCY_INDICES.values()), MULTI_COINCIDENT_WINDOW, 'mean'),
        # All market indices combined with majority vote
        (list(MARKET_INDICES.values()), MULTI_COINCIDENT_WINDOW, 'majority'),
        # All currency + market indices with correlation-weighted aggregation
        (list(CURRENCY_INDICES.values()) + list(MARKET_INDICES.values()), MULTI_COINCIDENT_WINDOW, 'correlation_weighted'),
    ],
    'correlation_threshold': 0.0,
}

# Multi-window returns strategy parameters
MULTI_WINDOW_PARAMS = {
    'enabled': True,
    'correlation_window': 60,  # Window for rolling correlation calculation (used in correlation_weighted aggregation)
    'same_asset_strategies': [
        (COINCIDENT_WINDOWS, 'weighted_average'),  # Short, medium, long-term weighted average
        (COINCIDENT_WINDOWS, 'majority_vote'),  # Short, medium, long-term majority vote
    ],
    'cross_asset_strategies': [
        ('SPY', COINCIDENT_WINDOWS, 'weighted_average'),
        ('QQQ', COINCIDENT_WINDOWS, 'majority_vote'),  # Custom windows for QQQ
    ],
}

# Hyperparameter grids for ML model tuning
ML_HYPERPARAMETER_GRIDS = {
    'Linear Regression': {},  # No hyperparameters to tune
    
    'Elastic Net': {
        'alpha': [0.01, 0.1, 1.0],
        'l1_ratio': [0.5, 0.9]
    },
    
    'LASSO': {
        'alpha': [0.01, 0.1, 1.0]
    },
    
    'Support Vector Machine': {
        'C': [1.0, 10.0],
        'gamma': ['scale'],
        'kernel': ['rbf']
    },
    
    'K-Nearest Neighbor': {
        'n_neighbors': [5, 10],
        'weights': ['uniform', 'distance']
    },
    
    'Decision Tree': {
        'max_depth': [1, 2, 3, 5],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 4],
    },
    
    'Extra Trees': {
        'n_estimators': [5, 10, 20, 100],
        'max_depth': [1, 2, 3, 5],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 4],
    },
    
    'Random Forest': {
        'n_estimators': [5, 10, 20, 100],
        'max_depth': [1, 2, 3, 5],
        'min_samples_split': [2, 10],
        'min_samples_leaf': [1, 4],
    },
    
    'Gradient Boosting': {
        'n_estimators': [5, 10, 20, 100],
        'learning_rate': [0.01, 0.1],
        'max_depth': [1, 2, 3, 5],
        'min_samples_split': [2, 10],
    },
    
    'Adaptive Boosting': {
        'n_estimators': [5, 10, 20, 100],
        'learning_rate': [0.1, 1.0]
    },
    
    'XGBoost': {
        'n_estimators': [5, 10, 20, 100],
        'learning_rate': [0.01, 0.1],
        'max_depth': [1, 2, 3, 5],
        'subsample': [0.8, 1.0],
    }
}

# Performance metrics
TRADING_DAYS_PER_YEAR = 252

# Imports


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import pickle
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, Optional, Iterable, List, Tuple, Any, Set
from abc import ABC, abstractmethod

# models
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso
# from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

#  plotting
from lets_plot import *
LetsPlot.setup_html()


class YFinanceCache:
    """Persistent cache for Yahoo Finance historical data."""
    
    def __init__(self, cache_dir='data/yfinance_cache'):
        """Initialize cache with storage directory.
        
        Args:
            cache_dir: Directory to store cached data files
        """
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
    
    def _get_cache_key(self, ticker, start_date, end_date):
        """Generate cache key from ticker and date range.
        
        Args:
            ticker: Ticker symbol
            start_date: Start date
            end_date: End date
            
        Returns:
            Cache key string
        """
        # Normalize ticker for filename (remove special chars)
        clean_ticker = ticker.replace('^', 'idx_').replace('=', '_').replace('-', '_').replace('.', '_')
        start_str = pd.Timestamp(start_date).strftime('%Y%m%d')
        end_str = pd.Timestamp(end_date).strftime('%Y%m%d')
        return f"{clean_ticker}_{start_str}_{end_str}"
    
    def get(self, ticker, start_date, end_date):
        """Retrieve cached data if available.
        
        Args:
            ticker: Ticker symbol
            start_date: Start date
            end_date: End date
            
        Returns:
            Cached Series or None if not found
        """
        cache_key = self._get_cache_key(ticker, start_date, end_date)
        cache_file = self.cache_dir / f"{cache_key}.pkl"
        
        if cache_file.exists():
            try:
                with open(cache_file, 'rb') as f:
                    data = pickle.load(f)
                # print(f"[CACHE] Loaded cached data for {ticker} from {cache_file.name}")
                return data
            except Exception as e:
                print(f"[CACHE] Warning: Failed to load cache for {ticker}: {e}")
                return None
        return None
    
    def set(self, ticker, start_date, end_date, data):
        """Store data in cache.
        
        Args:
            ticker: Ticker symbol
            start_date: Start date
            end_date: End date
            data: pandas Series to cache
        """
        cache_key = self._get_cache_key(ticker, start_date, end_date)
        cache_file = self.cache_dir / f"{cache_key}.pkl"
        
        try:
            with open(cache_file, 'wb') as f:
                pickle.dump(data, f)
            print(f"[CACHE] Saved data for {ticker} to {cache_file.name}")
        except Exception as e:
            print(f"[CACHE] Warning: Failed to save cache for {ticker}: {e}")
    
    def clear(self):
        """Clear all cached data."""
        for cache_file in self.cache_dir.glob('*.pkl'):
            cache_file.unlink()
        print(f"[CACHE] Cleared all cached data from {self.cache_dir}")


# Global cache instance
_cache = YFinanceCache()


def get_cached_ticker_data(ticker, start_date, end_date, fetch_func):
    """Get ticker data from cache or fetch if not available.
    
    Args:
        ticker: Ticker symbol
        start_date: Start date
        end_date: End date
        fetch_func: Function to fetch data if not cached (should return pandas Series or DataFrame)
        
    Returns:
        pandas Series or DataFrame of price data
    """
    # Try to get from cache first
    data = _cache.get(ticker, start_date, end_date)
    
    if data is not None:
        return data
    
    # Fetch fresh data
    print(f"[CACHE] Fetching fresh data for {ticker}...")
    data = fetch_func()
    
    # Store in cache
    if data is not None and not (isinstance(data, pd.Series) and data.empty) and not (isinstance(data, pd.DataFrame) and data.empty):
        _cache.set(ticker, start_date, end_date, data)
    
    return data


def clear_yfinance_cache():
    """Clear all Yahoo Finance cached data."""
    _cache.clear()


class DataLoader:
    """Handle data fetching and preprocessing."""
    
    def __init__(self, ticker, start_date, end_date):
        self.ticker = ticker
        self.start_date = start_date
        self.end_date = end_date
        self._data = None
    
    def load_data(self):
        """Download data from Yahoo Finance with persistent caching."""
        def _fetch():
            return yf.download(
                self.ticker,
                start=self.start_date,
                end=self.end_date,
                auto_adjust=True,
                progress=False  # Suppress progress bar
            )
        
        # Use cached data to ensure consistency across runs
        self._data = get_cached_ticker_data(
            self.ticker, 
            self.start_date, 
            self.end_date, 
            _fetch
        )
        return self._data
    
    def get_prices(self):
        """Return closing prices."""
        if self._data is None:
            self.load_data()
        return self._data['Close']
    
    def get_ohlcv(self):
        """Return OHLCV data."""
        if self._data is None:
            self.load_data()
        return self._data
    
    @staticmethod
    def clean_data(df, method='ffill'):
        """Handle missing values."""
        return df.fillna(method=method).dropna()

In [3]:
"""
STRATEGIES
"""

def sma(series: pd.Series, window: int) -> pd.Series:
    """Simple moving average."""
    return series.rolling(window=window, min_periods=int(window * 0.8)).mean()


def ema(series: pd.Series, span: int) -> pd.Series:
    """Exponential moving average."""
    return series.ewm(span=span, adjust=False).mean()


def macd(series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    """Return MACD line and signal line."""
    fast_ema = ema(series, fast)
    slow_ema = ema(series, slow)
    macd_line = fast_ema - slow_ema
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line


def macd_normalized(series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9, norm_window: int = 42):
    """Return normalized MACD line and signal line as z-scores.
    
    Transforms MACD to be stationary by computing rolling z-score,
    making it oscillate around 0 with typical range of [-3, 3].
    """
    macd_line, signal_line = macd(series, fast, slow, signal)
    
    # Normalize MACD line to z-score
    macd_zscore = zscore(macd_line, norm_window)
    signal_zscore = zscore(signal_line, norm_window)
    
    return macd_zscore, signal_zscore


def zscore(series: pd.Series, window: int = 42) -> pd.Series:
    """Z-score relative to rolling mean/std."""
    mu = series.rolling(window=window, min_periods=int(window * 0.8)).mean()
    sigma = series.rolling(window=window, min_periods=int(window * 0.8)).std()
    return (series - mu) / sigma


def rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """Compute the Relative Strength Index (RSI).

    Uses Wilder's smoothing method (as in the notebooks).
    Returns a pandas Series indexed like `series`.
    """
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Prepare series
    avg_gain = pd.Series(index=series.index, dtype=float)
    avg_loss = pd.Series(index=series.index, dtype=float)

    # First value: simple average
    if len(series) <= period:
        return pd.Series(np.nan, index=series.index)

    avg_gain.iloc[period] = gain.iloc[1 : period + 1].mean()
    avg_loss.iloc[period] = loss.iloc[1 : period + 1].mean()

    # Wilder smoothing
    for i in range(period + 1, len(series)):
        avg_gain.iloc[i] = (avg_gain.iloc[i - 1] * (period - 1) + gain.iloc[i]) / period
        avg_loss.iloc[i] = (avg_loss.iloc[i - 1] * (period - 1) + loss.iloc[i]) / period

    rs = avg_gain / avg_loss
    rsi_series = 100 - (100 / (1 + rs))
    return rsi_series


def stochastic_oscillator(high: pd.Series, low: pd.Series, close: pd.Series, 
                         k_period: int = 14, d_period: int = 3):
    """Compute Stochastic Oscillator %K and %D.
    
    %K = 100 * (Close - Lowest Low) / (Highest High - Lowest Low)
    %D = SMA of %K over d_period
    
    Returns: %K series, %D series
    """
    lowest_low = low.rolling(window=k_period, min_periods=int(k_period * 0.8)).min()
    highest_high = high.rolling(window=k_period, min_periods=int(k_period * 0.8)).max()
    
    k_percent = 100 * (close - lowest_low) / (highest_high - lowest_low)
    d_percent = k_percent.rolling(window=d_period, min_periods=int(d_period * 0.8)).mean()
    
    return k_percent, d_percent


def rate_of_change(series: pd.Series, period: int = 12) -> pd.Series:
    """Compute Rate of Change (ROC).
    
    ROC = 100 * (Price - Price_n_periods_ago) / Price_n_periods_ago
    """
    roc = 100 * (series - series.shift(period)) / series.shift(period)
    return roc


def cumulative_volume(volume: pd.Series) -> pd.Series:
    """Compute cumulative volume.
    
    Returns the cumulative sum of volume over time.
    """
    return volume.cumsum()


def on_balance_volume(close: pd.Series, volume: pd.Series) -> pd.Series:
    """Compute On-Balance Volume (OBV).
    
    Approximates buying/selling pressure based on price movement:
    - If close > previous close: add volume (buyers won the day)
    - If close < previous close: subtract volume (sellers won the day)
    - If close == previous close: no change
    
    This provides a running total of "signed" volume based on daily price direction.
    """
    obv = pd.Series(index=close.index, dtype=float)
    obv.iloc[0] = volume.iloc[0]
    
    for i in range(1, len(close)):
        if close.iloc[i] > close.iloc[i-1]:
            obv.iloc[i] = obv.iloc[i-1] + volume.iloc[i]
        elif close.iloc[i] < close.iloc[i-1]:
            obv.iloc[i] = obv.iloc[i-1] - volume.iloc[i]
        else:
            obv.iloc[i] = obv.iloc[i-1]
    
    return obv


def obv_normalized(close: pd.Series, volume: pd.Series, norm_window: int = 42) -> pd.Series:
    """Compute normalized On-Balance Volume (OBV) as z-score.
    
    Transforms OBV to be stationary by computing rolling z-score,
    making it oscillate around 0 with typical range of [-3, 3].
    """
    obv = on_balance_volume(close, volume)
    obv_zscore = zscore(obv, norm_window)
    return obv_zscore


def bollinger_bands(series: pd.Series, window: int = 20, num_std: float = 2.0):
    """Compute Bollinger Bands.
    
    Parameters
    ----------
    series : pd.Series
        Price series (typically closing prices)
    window : int
        Rolling window for mean and standard deviation
    num_std : float
        Number of standard deviations for upper/lower bands
    
    Returns
    -------
    tuple of (middle_band, upper_band, lower_band)
        middle_band: SMA of the price
        upper_band: SMA + (num_std * rolling std)
        lower_band: SMA - (num_std * rolling std)
    """
    middle_band = sma(series, window)
    rolling_std = series.rolling(window=window, min_periods=int(window * 0.8)).std()
    upper_band = middle_band + (num_std * rolling_std)
    lower_band = middle_band - (num_std * rolling_std)
    
    return middle_band, upper_band, lower_band

class TechnicalIndicators:
    """Class wrapper providing methods used by strategy classes.

    The notebooks/strategy code instantiate this class and call
    calculate_sma / calculate_ema / calculate_macd / calculate_zscore / calculate_rsi.
    """

    @staticmethod
    def calculate_sma(series: pd.Series, window: int) -> pd.Series:
        return sma(series, window)

    @staticmethod
    def calculate_ema(series: pd.Series, span: int) -> pd.Series:
        return ema(series, span)

    @staticmethod
    def calculate_macd(series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
        return macd(series, fast, slow, signal)
    
    @staticmethod
    def calculate_macd_normalized(series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9, norm_window: int = 42):
        return macd_normalized(series, fast, slow, signal, norm_window)

    @staticmethod
    def calculate_zscore(series: pd.Series, window: int = 42) -> pd.Series:
        return zscore(series, window)

    @staticmethod
    def calculate_rsi(series: pd.Series, period: int = 14) -> pd.Series:
        return rsi(series, period)
    
    @staticmethod
    def calculate_stochastic(high: pd.Series, low: pd.Series, close: pd.Series, 
                           k_period: int = 14, d_period: int = 3):
        return stochastic_oscillator(high, low, close, k_period, d_period)
    
    @staticmethod
    def calculate_roc(series: pd.Series, period: int = 12) -> pd.Series:
        return rate_of_change(series, period)
    
    @staticmethod
    def calculate_cumulative_volume(volume: pd.Series) -> pd.Series:
        return cumulative_volume(volume)
    
    @staticmethod
    def calculate_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
        return on_balance_volume(close, volume)
    
    @staticmethod
    def calculate_obv_normalized(close: pd.Series, volume: pd.Series, norm_window: int = 42) -> pd.Series:
        return obv_normalized(close, volume, norm_window)
    
    @staticmethod
    def calculate_bollinger_bands(series: pd.Series, window: int = 20, num_std: float = 2.0):
        return bollinger_bands(series, window, num_std)


class BaseStrategy(ABC):
    """Abstract base class for trading strategies."""
    
    def __init__(self, name, data):
        self.name = name
        self.data = data.copy()
        self.positions = None
        self.trades = None
        self.returns = None
    
    @abstractmethod
    def generate_signals(self):
        """Generate trading signals. Must be implemented by subclasses."""
        pass
    
    def calculate_returns(self):
        """Calculate strategy returns."""
        # Always regenerate signals to ensure consistency with current price data
        # This is important when strategies are reused across multiple runs or feature building
        self.generate_signals()
        
        # Calculate passive returns only once to avoid floating point drift
        # Check if passive_returns already exists and is valid
        if 'passive_returns' not in self.data.columns or self.data['passive_returns'].isna().all():
            # Passive returns - calculate once and cache
            price_col = self.data.columns[0]
            self.data['passive_returns'] = np.log(
                self.data[price_col] / 
                self.data[price_col].shift(1)
            ).fillna(0)
        
        # Strategy returns - always recalculate based on current positions
        self.data['strategy_returns'] = (
            self.data['passive_returns'] * 
            self.positions.shift(1).fillna(0)
        )
        
        # Cumulative returns
        self.data['cum_passive_returns'] = (
            self.data['passive_returns'].cumsum().apply(np.exp)
        )
        self.data['cum_strategy_returns'] = (
            self.data['strategy_returns'].cumsum().apply(np.exp)
        )
        
        return self.data
    
    def get_positions(self):
        """Return positions DataFrame."""
        if self.positions is None:
            self.generate_signals()
        return self.positions

class RSIStrategy(BaseStrategy):
    """RSI-based mean reversion strategy.

    This class implements a single permutation of RSI parameters (period,
    oversold, overbought). To create multiple permutations (different periods
    and thresholds), use the helper `generate_rsi_variants` provided below.
    """

    def __init__(self, data, period=14, oversold=30, overbought=70):
        # give each instance a descriptive name so backtests/comparisons are clear
        name = f"RSI_p{period}_os{oversold}_ob{overbought}"
        super().__init__(name, data)
        self.period = int(period)
        self.oversold = float(oversold)
        self.overbought = float(overbought)
    
    def generate_signals(self):
        """Generate RSI-based signals."""
        ti = TechnicalIndicators()
        
        self.data['rsi'] = ti.calculate_rsi(
            self.data[self.data.columns[0]], self.period
        )
        self.data = self.data.dropna()
        
        # Long when crossing above oversold, short when crossing below overbought
        self.positions = pd.Series(
            np.where(
                (self.data['rsi'].shift(1) < self.oversold) & 
                (self.data['rsi'] > self.oversold), 1,
                np.where(
                    (self.data['rsi'].shift(1) > self.overbought) & 
                    (self.data['rsi'] < self.overbought), -1, 
                    np.nan
                )
            ),
            index=self.data.index
        ).ffill().fillna(0)
        
        self.trades = self.positions.diff()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


class ZScoreStrategy(BaseStrategy):
    """Z-score based mean reversion strategy."""
    
    def __init__(self, data, window=42, threshold=2):
        super().__init__(f'ZScore_{window}', data)
        self.window = window
        self.threshold = threshold
    
    def generate_signals(self):
        """Generate Z-score based signals."""
        ti = TechnicalIndicators()
        
        self.data['zscore'] = ti.calculate_zscore(
            self.data[self.data.columns[0]], self.window
        )
        self.data = self.data.dropna()
        
        # Long when z < -threshold, short when z > threshold, exit at 0
        positions = np.where(
            self.data['zscore'] > self.threshold, -1,
            np.where(self.data['zscore'] < -self.threshold, 1, np.nan)
        )
        
        # Exit when crossing zero
        positions = np.where(
            self.data['zscore'] * self.data['zscore'].shift(1) < 0, 
            0, positions
        )
        
        self.positions = pd.Series(positions, index=self.data.index).ffill()
        self.trades = self.positions.diff()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


class BollingerBandStrategy(BaseStrategy):
    """Bollinger Band mean reversion strategy.
    
    This strategy uses Bollinger Bands to identify overbought and oversold conditions.
    It goes long when price touches the lower band and short when price touches the upper band,
    exiting when price returns to the middle band.
    """
    
    def __init__(self, data, window=20, num_std=2.0):
        name = f"BB_w{window}_std{num_std}"
        super().__init__(name, data)
        self.window = int(window)
        self.num_std = float(num_std)
    
    def generate_signals(self):
        """Generate Bollinger Band based signals."""
        ti = TechnicalIndicators()
        
        # Calculate Bollinger Bands
        middle, upper, lower = ti.calculate_bollinger_bands(
            self.data[self.data.columns[0]], 
            self.window, 
            self.num_std
        )
        
        self.data['bb_middle'] = middle
        self.data['bb_upper'] = upper
        self.data['bb_lower'] = lower
        self.data = self.data.dropna()
        
        price = self.data[self.data.columns[0]]
        
        # Signal logic:
        # Long when price crosses below lower band (oversold)
        # Short when price crosses above upper band (overbought)
        # Exit when price crosses middle band
        
        # Long signal: price touches or crosses below lower band
        long_entry = (price <= self.data['bb_lower'])
        # Short signal: price touches or crosses above upper band
        short_entry = (price >= self.data['bb_upper'])
        
        # Exit signals: price crosses middle band
        exit_long = (price >= self.data['bb_middle']) & (price.shift(1) < self.data['bb_middle'].shift(1))
        exit_short = (price <= self.data['bb_middle']) & (price.shift(1) > self.data['bb_middle'].shift(1))
        
        # Initialize positions
        positions = pd.Series(np.nan, index=self.data.index)
        
        # Set entry positions
        positions[long_entry] = 1
        positions[short_entry] = -1
        
        # Set exit positions
        positions[exit_long | exit_short] = 0
        
        # Forward fill to maintain positions
        self.positions = positions.ffill().fillna(0)
        self.trades = self.positions.diff()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


def generate_rsi_variants(data, periods=None, threshold_pairs=None):
    """Generate a list of RSIStrategy instances for all combinations of
    `periods` and `threshold_pairs`.

    Parameters
    ----------
    data : pd.Series or pd.DataFrame
        Price series or single-column DataFrame to pass to strategy instances.
    periods : list[int]
        List of RSI lookback periods to use. Default: [14, 21].
    threshold_pairs : list[tuple]
        List of (oversold, overbought) tuples. Default: [(30, 70), (25, 75)].

    Returns
    -------
    list[RSIStrategy]
        Strategy instances ready to be backtested.
    """

    if periods is None:
        periods = [14, 21]
    if threshold_pairs is None:
        threshold_pairs = [(30, 70), (25, 75)]

    variants = []
    # if user passed a Series, make it a DataFrame with a single column
    if isinstance(data, pd.Series):
        price_df = data.to_frame()
    else:
        price_df = data.copy()

    for p in periods:
        for (os_val, ob_val) in threshold_pairs:
            variants.append(RSIStrategy(price_df, period=p, oversold=os_val, overbought=ob_val))

    return variants

def generate_bollinger_variants(data, windows=None, std_devs=None):
    """Generate a list of BollingerBandStrategy instances for all combinations of
    `windows` and `std_devs`.

    Parameters
    ----------
    data : pd.Series or pd.DataFrame
        Price series or single-column DataFrame to pass to strategy instances.
    windows : list[int]
        List of Bollinger Band lookback windows to use. Default: [20, 30].
    std_devs : list[float]
        List of standard deviation multipliers. Default: [1.5, 2.0, 2.5].

    Returns
    -------
    list[BollingerBandStrategy]
        Strategy instances ready to be backtested.
    """
    import pandas as pd

    if windows is None:
        windows = [20, 30]
    if std_devs is None:
        std_devs = [1.5, 2.0, 2.5]

    variants = []
    # if user passed a Series, make it a DataFrame with a single column
    if isinstance(data, pd.Series):
        price_df = data.to_frame()
    else:
        price_df = data.copy()

    for w in windows:
        for std in std_devs:
            variants.append(BollingerBandStrategy(price_df, window=w, num_std=std))

    return variants


class SMAStrategy(BaseStrategy):
    """Simple Moving Average crossover strategy."""
    
    def __init__(self, data, short_window=20, long_window=60):
        super().__init__(f'SMA_{short_window}_{long_window}', data)
        self.short_window = short_window
        self.long_window = long_window
    
    def generate_signals(self):
        """Generate SMA crossover signals using normalized spread."""
        ti = TechnicalIndicators()
        
        price = self.data[self.data.columns[0]]
        
        # Calculate SMAs
        sma_short = ti.calculate_sma(price, self.short_window)
        sma_long = ti.calculate_sma(price, self.long_window)
        
        # Calculate spread and normalize to z-score for stationarity
        spread = sma_short - sma_long
        spread_zscore = ti.calculate_zscore(spread, window=42)
        
        self.data[f'sma_{self.short_window}'] = sma_short
        self.data[f'sma_{self.long_window}'] = sma_long
        self.data['spread_zscore'] = spread_zscore
        
        self.data = self.data.dropna()
        
        # Use z-score of spread: long when > 0, short when < 0
        self.positions = pd.Series(
            np.where(self.data['spread_zscore'] > 0, 1, -1),
            index=self.data.index
        )
        
        self.trades = self.positions.diff()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


class MACDStrategy(BaseStrategy):
    """MACD crossover strategy."""
    
    def __init__(self, data, fast=12, slow=26, signal=9):
        super().__init__(f'MACD_{fast}_{slow}_{signal}', data)
        self.fast = fast
        self.slow = slow
        self.signal = signal
    
    def generate_signals(self):
        """Generate MACD crossover signals."""
        ti = TechnicalIndicators()
        
        # Use normalized MACD for stationary oscillator
        macd, signal_line = ti.calculate_macd_normalized(
            self.data[self.data.columns[0]], 
            self.fast, self.slow, self.signal,
            norm_window=42  # Z-score normalization window
        )
        
        self.data['macd'] = macd
        self.data['signal_line'] = signal_line
        self.data = self.data.dropna()
        
        self.positions = pd.Series(
            np.where(self.data['macd'] > self.data['signal_line'], 1, -1),
            index=self.data.index
        )
        
        self.trades = self.positions.diff()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


class StochasticStrategy(BaseStrategy):
    """Stochastic Oscillator (%K, %D) strategy.
    
    Goes long when %K crosses above %D and both are below oversold threshold,
    goes short when %K crosses below %D and both are above overbought threshold.
    """
    
    def __init__(self, data, ticker='SPY', k_period=14, d_period=3, 
                 oversold=20, overbought=80):
        """
        Args:
            data: Price DataFrame
            ticker: Ticker symbol to fetch OHLC data for
            k_period: Period for %K calculation
            d_period: Period for %D (SMA of %K)
            oversold: Oversold threshold (default 20)
            overbought: Overbought threshold (default 80)
        """
        super().__init__(f'Stoch_{ticker}_k{k_period}_d{d_period}_os{oversold}_ob{overbought}', data)
        self.ticker = ticker
        self.k_period = k_period
        self.d_period = d_period
        self.oversold = oversold
        self.overbought = overbought
        self.ohlc_data = None
    
    def fetch_ohlc_data(self, start_date, end_date):
        """Fetch OHLC data for the ticker."""
        try:
            ticker_obj = yf.Ticker(self.ticker)
            hist_data = ticker_obj.history(start=start_date, end=end_date)
            if hist_data.empty:
                print(f"Warning: No OHLC data found for {self.ticker}")
                return None
            # Remove timezone to match price data
            if hist_data.index.tz is not None:
                hist_data.index = hist_data.index.tz_localize(None)
            return hist_data[['High', 'Low', 'Close']]
        except Exception as e:
            print(f"Error fetching OHLC data for {self.ticker}: {e}")
            return None
    
    def generate_signals(self):
        """Generate Stochastic Oscillator signals."""
        # Fetch OHLC data
        start_date = self.data.index[0] - pd.Timedelta(days=self.k_period * 2)
        end_date = self.data.index[-1]
        
        self.ohlc_data = self.fetch_ohlc_data(start_date, end_date)
        
        if self.ohlc_data is None or self.ohlc_data.empty:
            print(f"Warning: Could not fetch OHLC data for {self.ticker}. Using neutral positions.")
            self.positions = pd.Series(0, index=self.data.index)
            self.data['positions'] = self.positions
            self.data['trades'] = 0
            return self.positions
        
        # Align OHLC data with price data
        self.ohlc_data = self.ohlc_data.reindex(self.data.index, method='ffill')
        
        ti = TechnicalIndicators()
        
        # Calculate %K and %D
        k_percent, d_percent = ti.calculate_stochastic(
            self.ohlc_data['High'],
            self.ohlc_data['Low'],
            self.ohlc_data['Close'],
            self.k_period,
            self.d_period
        )
        
        # Store in data
        self.data['k_percent'] = k_percent
        self.data['d_percent'] = d_percent
        
        # Generate signals
        # Long when %K crosses above %D in oversold region
        # Short when %K crosses below %D in overbought region
        k_above_d = k_percent > d_percent
        k_below_d = k_percent < d_percent
        
        in_oversold = (k_percent < self.oversold) & (d_percent < self.oversold)
        in_overbought = (k_percent > self.overbought) & (d_percent > self.overbought)
        
        self.positions = pd.Series(0, index=self.data.index)
        self.positions[k_above_d & in_oversold] = 1
        self.positions[k_below_d & in_overbought] = -1
        
        # Forward fill positions
        self.positions = self.positions.replace(0, np.nan).ffill().fillna(0)
        
        self.trades = self.positions.diff().abs()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


class ROCStrategy(BaseStrategy):
    """Rate of Change (ROC) momentum strategy.
    
    Goes long when ROC is positive and accelerating,
    goes short when ROC is negative and decelerating.
    """
    
    def __init__(self, data, period=12, threshold=0):
        """
        Args:
            data: Price DataFrame
            period: Lookback period for ROC calculation
            threshold: Threshold for signal generation (default 0)
        """
        super().__init__(f'ROC_p{period}_th{threshold}', data)
        self.period = period
        self.threshold = threshold
    
    def generate_signals(self):
        """Generate ROC signals."""
        ti = TechnicalIndicators()
        
        price_col = self.data.columns[0]
        
        # Calculate ROC
        roc = ti.calculate_roc(self.data[price_col], self.period)
        
        self.data['roc'] = roc
        self.data = self.data.dropna()
        
        # Generate positions: long when ROC > threshold, short when ROC < -threshold
        # Use the cleaned data index after dropna
        self.positions = pd.Series(
            np.where(self.data['roc'] > self.threshold, 1, 
                    np.where(self.data['roc'] < -self.threshold, -1, 0)),
            index=self.data.index
        )
        
        # Forward fill to avoid too many neutral positions
        self.positions = self.positions.replace(0, np.nan).ffill().fillna(0)
        
        self.trades = self.positions.diff().abs()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


class OBVStrategy(BaseStrategy):
    """Strategy based on On-Balance Volume (OBV).
    
    Uses price direction to estimate buying/selling pressure:
    - When close > previous close: add volume to OBV (accumulation)
    - When close < previous close: subtract volume from OBV (distribution)
    
    Goes long when OBV is trending up (above its moving average),
    short when trending down.
    """
    
    def __init__(self, data, ticker='SPY', window=20):
        """
        Args:
            data: Price DataFrame
            ticker: Ticker symbol to fetch OHLC data for
            window: Window for moving average of OBV
        """
        super().__init__(f'OBV_{ticker}_{window}', data)
        self.ticker = ticker
        self.window = window
        self.ticker_data = None
        
    def fetch_ticker_data(self, start_date, end_date):
        """Fetch OHLC data for the ticker."""
        try:
            ticker_obj = yf.Ticker(self.ticker)
            hist_data = ticker_obj.history(start=start_date, end=end_date)
            if hist_data.empty:
                print(f"Warning: No data found for {self.ticker}")
                return None
            # Remove timezone to match price data
            if hist_data.index.tz is not None:
                hist_data.index = hist_data.index.tz_localize(None)
            return hist_data
        except Exception as e:
            print(f"Error fetching data for {self.ticker}: {e}")
            return None
    
    def generate_signals(self):
        """Generate signals based on OBV trend."""
        # Fetch ticker data
        start_date = self.data.index[0] - pd.Timedelta(days=self.window * 2)
        end_date = self.data.index[-1]
        
        self.ticker_data = self.fetch_ticker_data(start_date, end_date)
        
        if self.ticker_data is None or self.ticker_data.empty:
            print(f"Warning: Could not fetch data for {self.ticker}. Using neutral positions.")
            self.positions = pd.Series(0, index=self.data.index)
            self.data['positions'] = self.positions
            self.data['trades'] = 0
            return self.positions
        
        # Align ticker data with price data
        self.ticker_data = self.ticker_data.reindex(self.data.index, method='ffill')
        
        ti = TechnicalIndicators()
        
        # Extract close and volume
        close = self.ticker_data['Close']
        volume = self.ticker_data['Volume']
        
        # Calculate normalized OBV (z-score for stationarity)
        obv_norm = ti.calculate_obv_normalized(close, volume, norm_window=42)
        
        # Store in data
        self.data['volume'] = volume
        self.data['obv_zscore'] = obv_norm
        
        # Generate positions: long when OBV z-score > 0, short when below
        self.positions = pd.Series(
            np.where(obv_norm > 0, 1, -1),
            index=self.data.index
        )
        
        self.positions = self.positions.fillna(0)
        
        self.trades = self.positions.diff().abs()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions



In [4]:
"""
Indices and Currencies
"""

class CoincidentIndexStrategy(BaseStrategy):
    """Strategy based on coincident economic/market indices.
    
    Uses price movements or returns from correlated assets (e.g., DXY for USD strength,
    VIX for volatility, GLD for gold, TLT for bonds) to generate trading signals.
    """
    
    def __init__(self, data, coincident_ticker, window=20, correlation_threshold=0.0, name_suffix=""):
        """
        Args:
            data: Price DataFrame for primary asset
            coincident_ticker: Ticker symbol for coincident index (e.g., 'DX-Y.NYB', 'GLD', '^VIX')
            window: Lookback window for calculating correlations/signals
            correlation_threshold: Threshold for signal generation based on coincident index returns
            name_suffix: Optional suffix for strategy name
        """
        strategy_name = f'Coincident_{coincident_ticker}_{window}' + (f'_{name_suffix}' if name_suffix else '')
        super().__init__(strategy_name, data)
        self.coincident_ticker = coincident_ticker
        self.window = window
        self.correlation_threshold = correlation_threshold
        self.coincident_data = None
        self._cached_raw_data = None  # Cache for Yahoo Finance data to ensure consistency
        
    def fetch_coincident_data(self, start_date, end_date):
        """Fetch data for the coincident index."""
        # Return in-memory cached data if available (for multiple calls within same run)
        if self._cached_raw_data is not None:
            return self._cached_raw_data
        
        # Use persistent disk cache to ensure consistency across runs
        def _fetch():
            ticker = yf.Ticker(self.coincident_ticker)
            coincident = ticker.history(start=start_date, end=end_date)
            if coincident.empty:
                print(f"Warning: No data found for {self.coincident_ticker}")
                return None
            # Remove timezone to match primary data
            close_data = coincident['Close']
            if close_data.index.tz is not None:
                close_data.index = close_data.index.tz_localize(None)
            return close_data
        
        try:
            close_data = get_cached_ticker_data(
                self.coincident_ticker, start_date, end_date, _fetch
            )
            # Cache in memory for subsequent calls within same run
            self._cached_raw_data = close_data
            return close_data
        except Exception as e:
            print(f"Error fetching {self.coincident_ticker}: {e}")
            return None
    
    def generate_signals(self):
        """Generate signals based on coincident index momentum."""
        # Determine date range from primary data
        start_date = self.data.index[0] - pd.Timedelta(days=self.window * 2)
        end_date = self.data.index[-1]
        
        # Fetch coincident data
        self.coincident_data = self.fetch_coincident_data(start_date, end_date)
        
        if self.coincident_data is None or self.coincident_data.empty:
            print(f"Warning: Could not fetch {self.coincident_ticker} data. Using neutral positions.")
            self.positions = pd.Series(0, index=self.data.index)
            self.data['positions'] = self.positions
            self.data['trades'] = 0
            return self.positions
        
        # Align coincident data with primary data index
        self.coincident_data = self.coincident_data.reindex(self.data.index, method='ffill')
        
        # Calculate returns for coincident index
        coincident_returns = np.log(self.coincident_data / self.coincident_data.shift(1))
        
        # FIX DATA LEAKAGE: Shift by 1 to use only PAST data
        # Signal at time t should only use data up to time t-1 (not including t)
        # This ensures we don't use today's close price (which isn't known until after market close)
        coincident_signal = coincident_returns.shift(1).rolling(window=self.window).mean()
        
        # Store in data for analysis
        self.data[f'{self.coincident_ticker}_price'] = self.coincident_data
        self.data[f'{self.coincident_ticker}_signal'] = coincident_signal
        
        # Add z-score of price for ML features (stationary version)
        # Also shift to avoid leakage
        ti = TechnicalIndicators()
        self.data[f'{self.coincident_ticker}_price_zscore'] = ti.calculate_zscore(
            self.coincident_data.shift(1), window=self.window
        )
        
        # Generate positions: long when coincident index momentum is positive, short when negative
        self.positions = pd.Series(
            np.where(coincident_signal > self.correlation_threshold, 1, -1),
            index=self.data.index
        )
        
        # Fill any NaN positions with 0 (neutral)
        self.positions = self.positions.fillna(0)
        
        self.trades = self.positions.diff().abs()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


class MultiCoincidentStrategy(BaseStrategy):
    """Strategy combining multiple coincident indices.
    
    Aggregates signals from multiple correlated assets to generate a composite signal.
    Can also optionally exclude individual ticker features to avoid redundancy.
    """
    
    def __init__(self, data, coincident_tickers, window=20, aggregation='mean', 
                 include_individual_features=True, correlation_window=60, name_suffix=""):
        """
        Args:
            data: Price DataFrame for primary asset
            coincident_tickers: List of ticker symbols for coincident indices
            window: Lookback window for calculating signals
            aggregation: How to combine signals ('mean', 'sum', 'majority', 'correlation_weighted')
            include_individual_features: If False, only composite_signal is kept for ML features
            correlation_window: Window for calculating correlations (used for 'correlation_weighted')
            name_suffix: Optional suffix for strategy name
        """
        tickers_str = '_'.join([t.replace('^', '').replace('-', '')[:3] for t in coincident_tickers])
        strategy_name = f'MultiCoinc_{tickers_str}_{window}' + (f'_{name_suffix}' if name_suffix else '')
        super().__init__(strategy_name, data)
        self.coincident_tickers = coincident_tickers
        self.window = window
        self.aggregation = aggregation
        self.include_individual_features = include_individual_features
        self.correlation_window = correlation_window
        self._cached_raw_data = None  # Cache for Yahoo Finance data to ensure consistency
        
    def fetch_coincident_data(self, start_date, end_date):
        """Fetch data for all coincident indices."""
        # Return in-memory cached data if available (for multiple calls within same run)
        if self._cached_raw_data is not None:
            return self._cached_raw_data
        
        coincident_dict = {}
        for ticker in self.coincident_tickers:
            # Use persistent disk cache for each ticker
            def _fetch():
                ticker_obj = yf.Ticker(ticker)
                ticker_data = ticker_obj.history(start=start_date, end=end_date)
                if not ticker_data.empty:
                    close_data = ticker_data['Close']
                    # Remove timezone to match primary data
                    if close_data.index.tz is not None:
                        close_data.index = close_data.index.tz_localize(None)
                    return close_data
                else:
                    print(f"Warning: No data found for {ticker}")
                    return None
            
            try:
                data = get_cached_ticker_data(ticker, start_date, end_date, _fetch)
                if data is not None:
                    coincident_dict[ticker] = data
            except Exception as e:
                print(f"Error fetching {ticker}: {e}")
        
        if not coincident_dict:
            return None
        
        result = pd.DataFrame(coincident_dict)
        # Cache in memory for subsequent calls within same run
        self._cached_raw_data = result
        return result
    
    def generate_signals(self):
        """Generate composite signals from multiple coincident indices."""
        # Determine date range
        start_date = self.data.index[0] - pd.Timedelta(days=self.window * 2)
        end_date = self.data.index[-1]
        
        # Fetch all coincident data
        coincident_df = self.fetch_coincident_data(start_date, end_date)
        
        if coincident_df is None or coincident_df.empty:
            print(f"Warning: Could not fetch coincident data. Using neutral positions.")
            self.positions = pd.Series(0, index=self.data.index)
            self.data['positions'] = self.positions
            self.data['trades'] = 0
            return self.positions
        
        # Align with primary data index
        coincident_df = coincident_df.reindex(self.data.index, method='ffill')
        
        # Calculate SPY returns for correlation weighting
        price_col = self.data.columns[0]  # First column is price
        spy_returns = np.log(self.data[price_col] / self.data[price_col].shift(1))
        
        # Calculate signals for each coincident index
        signals = pd.DataFrame(index=self.data.index)
        correlations = pd.DataFrame(index=self.data.index)
        
        for ticker in coincident_df.columns:
            # Calculate returns
            returns = np.log(coincident_df[ticker] / coincident_df[ticker].shift(1))
            # FIX DATA LEAKAGE: Shift by 1 to use only PAST data
            # Signal at time t should only use data up to time t-1
            signal = returns.shift(1).rolling(window=self.window).mean()
            # Convert to position: 1 if positive momentum, -1 if negative
            signals[ticker] = np.where(signal > 0, 1, -1)
            
            # Calculate rolling correlation for weighting (also shifted to avoid leakage)
            if self.aggregation == 'correlation_weighted':
                # Use shifted returns to avoid leakage
                corr = returns.shift(1).rolling(window=self.correlation_window).corr(spy_returns.shift(1))
                # Use absolute correlation for weighting (both positive and negative correlations are useful)
                correlations[ticker] = corr.abs()
            
            # Only store individual features if requested (avoid redundancy with CoincidentIndexStrategy)
            if self.include_individual_features:
                # Store raw data
                self.data[f'{ticker}_price'] = coincident_df[ticker]
                self.data[f'{ticker}_signal'] = signal
                
                # Add z-score of price for ML features (stationary version)
                # Also shift to avoid leakage
                from features.technical_indicators import TechnicalIndicators
                ti = TechnicalIndicators()
                self.data[f'{ticker}_price_zscore'] = ti.calculate_zscore(
                    coincident_df[ticker].shift(1), window=self.window
                )
        
        # Aggregate signals
        if self.aggregation == 'mean':
            # Average of all signals (can be fractional)
            composite_signal = signals.mean(axis=1)
            self.positions = pd.Series(
                np.where(composite_signal > 0, 1, -1),
                index=self.data.index
            )
        elif self.aggregation == 'sum':
            # Sum of all signals
            composite_signal = signals.sum(axis=1)
            self.positions = pd.Series(
                np.where(composite_signal > 0, 1, -1),
                index=self.data.index
            )
        elif self.aggregation == 'majority':
            # Majority vote
            composite_signal = signals.sum(axis=1)
            self.positions = pd.Series(
                np.where(composite_signal > 0, 1, 
                        np.where(composite_signal < 0, -1, 0)),
                index=self.data.index
            )
        elif self.aggregation == 'correlation_weighted':
            # Weighted by historical correlation to SPY returns
            # Normalize correlations to sum to 1 for each row (avoiding division by zero)
            weights = correlations.div(correlations.sum(axis=1), axis=0)
            weights = weights.fillna(1.0 / len(self.coincident_tickers))  # Equal weight if no correlation data
            
            # Weight the signals by correlation
            weighted_signals = signals * weights
            composite_signal = weighted_signals.sum(axis=1)
            
            self.positions = pd.Series(
                np.where(composite_signal > 0, 1, -1),
                index=self.data.index
            )
        else:
            raise ValueError(f"Unknown aggregation method: {self.aggregation}")
        
        self.data['composite_signal'] = composite_signal
        self.positions = self.positions.fillna(0)
        
        self.trades = self.positions.diff().abs()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions



In [5]:
"""
Calculate returns over multiple time windows
"""
class MultiWindowReturnsStrategy(BaseStrategy):
    """Strategy based on returns calculated over multiple time windows.
    
    Computes returns over various lookback periods and generates signals based on
    momentum patterns across different time horizons.
    """
    
    def __init__(self, data, windows=[5, 10, 20, 60], signal_method='weighted_average', name_suffix=""):
        """
        Args:
            data: Price DataFrame for the asset
            windows: List of lookback windows (in days) for computing returns
            signal_method: Method to generate signals from multi-window returns
                          ('weighted_average', 'majority_vote', 'all_positive', 'trend_alignment')
            name_suffix: Optional suffix for strategy name
        """
        windows_str = '_'.join(map(str, windows))
        strategy_name = f'MultiWindow_{signal_method}_{windows_str}' + (f'_{name_suffix}' if name_suffix else '')
        super().__init__(strategy_name, data)
        self.windows = sorted(windows)
        self.signal_method = signal_method
        
    def generate_signals(self):
        """Generate signals based on multi-window returns."""
        price_col = self.data.columns[0]
        
        # Calculate returns for each window
        returns_dict = {}
        for window in self.windows:
            # Log returns over the lookback window
            ret = np.log(self.data[price_col] / self.data[price_col].shift(window))
            returns_dict[f'ret_{window}d'] = ret
            self.data[f'ret_{window}d'] = ret
        
        returns_df = pd.DataFrame(returns_dict, index=self.data.index)
        
        # Generate signals based on chosen method
        if self.signal_method == 'weighted_average':
            # Weight longer-term returns more heavily
            weights = np.array([1/w for w in self.windows])
            weights = weights / weights.sum()  # Normalize
            
            weighted_signal = sum(returns_df[f'ret_{w}d'] * weight 
                                for w, weight in zip(self.windows, weights))
            
            self.positions = pd.Series(
                np.where(weighted_signal > 0, 1, -1),
                index=self.data.index
            )
            self.data['weighted_signal'] = weighted_signal
            
        elif self.signal_method == 'majority_vote':
            # Each window votes: +1 for positive return, -1 for negative
            votes = pd.DataFrame(index=self.data.index)
            for window in self.windows:
                votes[f'vote_{window}'] = np.where(returns_df[f'ret_{window}d'] > 0, 1, -1)
            
            majority_signal = votes.sum(axis=1)
            self.positions = pd.Series(
                np.where(majority_signal > 0, 1, -1),
                index=self.data.index
            )
            self.data['majority_signal'] = majority_signal
            
        elif self.signal_method == 'all_positive':
            # Go long only if ALL windows show positive returns, short if ALL negative
            all_positive = returns_df.gt(0).all(axis=1)
            all_negative = returns_df.lt(0).all(axis=1)
            
            self.positions = pd.Series(
                np.where(all_positive, 1, 
                        np.where(all_negative, -1, 0)),
                index=self.data.index
            )
            self.data['all_positive'] = all_positive
            self.data['all_negative'] = all_negative
            
        elif self.signal_method == 'trend_alignment':
            # Check if windows show aligned trend (short < medium < long term returns)
            # This indicates accelerating momentum
            if len(self.windows) >= 3:
                # Check if returns are monotonically increasing across windows
                aligned_up = True
                aligned_down = True
                
                for i in range(len(self.windows) - 1):
                    curr_col = f'ret_{self.windows[i]}d'
                    next_col = f'ret_{self.windows[i+1]}d'
                    
                    if i == 0:
                        aligned_up = returns_df[curr_col] < returns_df[next_col]
                        aligned_down = returns_df[curr_col] > returns_df[next_col]
                    else:
                        aligned_up = aligned_up & (returns_df[curr_col] < returns_df[next_col])
                        aligned_down = aligned_down & (returns_df[curr_col] > returns_df[next_col])
                
                self.positions = pd.Series(
                    np.where(aligned_up, 1, 
                            np.where(aligned_down, -1, 0)),
                    index=self.data.index
                )
                self.data['aligned_up'] = aligned_up
                self.data['aligned_down'] = aligned_down
            else:
                # Fall back to simple average if not enough windows
                avg_signal = returns_df.mean(axis=1)
                self.positions = pd.Series(
                    np.where(avg_signal > 0, 1, -1),
                    index=self.data.index
                )
                self.data['avg_signal'] = avg_signal
        else:
            raise ValueError(f"Unknown signal_method: {self.signal_method}")
        
        # Fill NaN positions with 0 (neutral)
        self.positions = self.positions.fillna(0)
        
        self.trades = self.positions.diff().abs()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


class CrossAssetWindowReturnsStrategy(BaseStrategy):
    """Strategy using multi-window returns from a different asset as signals.
    
    Similar to MultiWindowReturnsStrategy but uses returns from a coincident asset
    (like SPY, QQQ, or sector ETFs) to generate signals for the primary asset.
    """
    
    def __init__(self, data, reference_ticker='SPY', windows=[5, 10, 20, 60], 
                 signal_method='weighted_average', name_suffix=""):
        """
        Args:
            data: Price DataFrame for primary asset
            reference_ticker: Ticker symbol for reference asset (e.g., 'SPY', 'QQQ')
            windows: List of lookback windows (in days) for computing returns
            signal_method: Method to generate signals from multi-window returns
            name_suffix: Optional suffix for strategy name
        """
        windows_str = '_'.join(map(str, windows))
        strategy_name = f'CrossAsset_{reference_ticker}_{signal_method}_{windows_str}' + (f'_{name_suffix}' if name_suffix else '')
        super().__init__(strategy_name, data)
        self.reference_ticker = reference_ticker
        self.windows = sorted(windows)
        self.signal_method = signal_method
        self.reference_data = None
        
    def fetch_reference_data(self, start_date, end_date):
        """Fetch data for the reference asset."""
        try:
            ticker = yf.Ticker(self.reference_ticker)
            ref_data = ticker.history(start=start_date, end=end_date)
            if ref_data.empty:
                print(f"Warning: No data found for {self.reference_ticker}")
                return None
            # Remove timezone to match primary data
            close_data = ref_data['Close']
            if close_data.index.tz is not None:
                close_data.index = close_data.index.tz_localize(None)
            return close_data
        except Exception as e:
            print(f"Error fetching {self.reference_ticker}: {e}")
            return None
    
    def generate_signals(self):
        """Generate signals based on reference asset's multi-window returns."""
        # Determine date range (add buffer for longest window)
        max_window = max(self.windows)
        start_date = self.data.index[0] - pd.Timedelta(days=max_window * 2)
        end_date = self.data.index[-1]
        
        # Fetch reference data
        self.reference_data = self.fetch_reference_data(start_date, end_date)
        
        if self.reference_data is None or self.reference_data.empty:
            print(f"Warning: Could not fetch {self.reference_ticker} data. Using neutral positions.")
            self.positions = pd.Series(0, index=self.data.index)
            self.data['positions'] = self.positions
            self.data['trades'] = 0
            return self.positions
        
        # Align reference data with primary data index
        self.reference_data = self.reference_data.reindex(self.data.index, method='ffill')
        self.data[f'{self.reference_ticker}_price'] = self.reference_data
        
        # Calculate returns for each window on reference asset
        returns_dict = {}
        for window in self.windows:
            ret = np.log(self.reference_data / self.reference_data.shift(window))
            returns_dict[f'{self.reference_ticker}_ret_{window}d'] = ret
            self.data[f'{self.reference_ticker}_ret_{window}d'] = ret
        
        returns_df = pd.DataFrame(returns_dict, index=self.data.index)
        
        # Generate signals using same logic as MultiWindowReturnsStrategy
        if self.signal_method == 'weighted_average':
            weights = np.array([1/w for w in self.windows])
            weights = weights / weights.sum()
            
            weighted_signal = sum(returns_df.iloc[:, i] * weight 
                                for i, weight in enumerate(weights))
            
            self.positions = pd.Series(
                np.where(weighted_signal > 0, 1, -1),
                index=self.data.index
            )
            self.data['weighted_signal'] = weighted_signal
            
        elif self.signal_method == 'majority_vote':
            votes = (returns_df > 0).astype(int) * 2 - 1  # Convert True/False to 1/-1
            majority_signal = votes.sum(axis=1)
            
            self.positions = pd.Series(
                np.where(majority_signal > 0, 1, -1),
                index=self.data.index
            )
            self.data['majority_signal'] = majority_signal
            
        elif self.signal_method == 'all_positive':
            all_positive = returns_df.gt(0).all(axis=1)
            all_negative = returns_df.lt(0).all(axis=1)
            
            self.positions = pd.Series(
                np.where(all_positive, 1, 
                        np.where(all_negative, -1, 0)),
                index=self.data.index
            )
            
        else:  # Default to simple average
            avg_signal = returns_df.mean(axis=1)
            self.positions = pd.Series(
                np.where(avg_signal > 0, 1, -1),
                index=self.data.index
            )
            self.data['avg_signal'] = avg_signal
        
        self.positions = self.positions.fillna(0)
        
        self.trades = self.positions.diff().abs()
        self.data['positions'] = self.positions
        self.data['trades'] = self.trades
        
        return self.positions


In [6]:
"""
Performance Metrics
"""
class PerformanceMetrics:
    """Calculate various performance metrics for strategies."""
    
    @staticmethod
    def calculate_sharpe_ratio(returns, periods_per_year=252):
        """Calculate annualized Sharpe ratio."""
        return np.sqrt(periods_per_year) * returns.mean() / returns.std(ddof=1)
    
    @staticmethod
    def calculate_cagr(cum_returns):
        """Calculate Compound Annual Growth Rate."""
        cum_returns = cum_returns.dropna()
        n_days = (cum_returns.index[-1] - cum_returns.index[0]).days
        cagr = (cum_returns.iloc[-1] / cum_returns.iloc[0]) ** (365.25/n_days) - 1
        return cagr
    
    @staticmethod
    def calculate_max_drawdown(cum_returns):
        """Calculate maximum drawdown."""
        drawdown = cum_returns / cum_returns.cummax() - 1
        return drawdown.min()
    
    @staticmethod
    def calculate_longest_drawdown_duration(cum_returns):
        """Calculate longest drawdown period in days."""
        drawdown = cum_returns / cum_returns.cummax() - 1
        periods = np.diff(
            np.append(drawdown[drawdown == 0].index, drawdown.index[-1:])
        )
        return periods.max() / np.timedelta64(1, "D")
    
    @staticmethod
    def get_drawdown_periods(cum_returns):
        """Get detailed drawdown period statistics."""
        drawdown = cum_returns / cum_returns.cummax() - 1
        dd_reset = pd.DataFrame({'Date': cum_returns.index, 'dd': drawdown.values})
        dd_reset['period'] = (dd_reset['dd'] == 0).cumsum()
        
        dd_nonzero = dd_reset[dd_reset['dd'] != 0]
        period_stats = dd_nonzero.groupby('period').agg(
            start_date=('Date', 'min'),
            end_date=('Date', 'max'),
            max_dd=('dd', 'min'),
            duration=('dd', 'count')
        ).sort_values('max_dd')
        
        return period_stats
    
    @staticmethod
    def calculate_all_metrics(strategy_returns, cum_returns, train_mse=None, test_mse=None, final_value=None):
        """Calculate all performance metrics."""
        pm = PerformanceMetrics()
        
        metrics = {
            'Sharpe Ratio': pm.calculate_sharpe_ratio(strategy_returns),
            'CAGR': pm.calculate_cagr(cum_returns),
            'Max Drawdown': pm.calculate_max_drawdown(cum_returns),
            'Longest DD Duration': pm.calculate_longest_drawdown_duration(cum_returns),
            'Total Return': cum_returns.iloc[-1] - 1,
            'Volatility': strategy_returns.std() * np.sqrt(252),
            'Final Value': final_value,
            'Train MSE': train_mse,
            'Test MSE': test_mse,
        }
        
        return metrics

In [7]:
"""
Checkpoint Management for model training
"""

def get_checkpoint_info(checkpoint_dir='checkpoints'):
    """Get information about the current checkpoint.
    
    Returns:
        dict with checkpoint info or None if no checkpoint exists
    """
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_file = checkpoint_path / 'ml_strategies_checkpoint.pkl'
    
    if not checkpoint_file.exists():
        return None
    
    try:
        with open(checkpoint_file, 'rb') as f:
            checkpoint_data = pickle.load(f)
        
        return {
            'num_strategies': len(checkpoint_data.get('strategies', [])),
            'completed_models': list(checkpoint_data.get('completed_models', [])),
            'timestamp': checkpoint_data.get('timestamp'),
            'file_size': checkpoint_file.stat().st_size / 1024,  # KB
        }
    except Exception as e:
        return {'error': str(e)}


def clear_checkpoint(checkpoint_dir='checkpoints'):
    """Clear the checkpoint file to start fresh.
    
    Returns:
        bool: True if checkpoint was cleared, False if no checkpoint existed
    """
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_file = checkpoint_path / 'ml_strategies_checkpoint.pkl'
    
    if checkpoint_file.exists():
        try:
            checkpoint_file.unlink()
            print(f"✓ Checkpoint cleared: {checkpoint_file}")
            return True
        except Exception as e:
            print(f"✗ Error clearing checkpoint: {e}")
            return False
    else:
        print("No checkpoint file found.")
        return False


def show_checkpoint_status(checkpoint_dir='checkpoints'):
    """Display current checkpoint status."""
    info = get_checkpoint_info(checkpoint_dir)
    
    if info is None:
        print("=" * 60)
        print("CHECKPOINT STATUS")
        print("=" * 60)
        print("No checkpoint found - starting fresh")
        print("=" * 60)
        return
    
    if 'error' in info:
        print("=" * 60)
        print("CHECKPOINT STATUS")
        print("=" * 60)
        print(f"Error reading checkpoint: {info['error']}")
        print("=" * 60)
        return
    
    print("=" * 60)
    print("CHECKPOINT STATUS")
    print("=" * 60)
    print(f"Checkpoint found: {info['num_strategies']} strategies completed")
    print(f"File size: {info['file_size']:.2f} KB")
    if info['timestamp']:
        print(f"Last updated: {info['timestamp']}")
    print()
    print("Completed models:")
    for model in sorted(info['completed_models']):
        print(f"  ✓ {model}")
    print("=" * 60)
    print("You can:")
    print("  - Run main.py to resume from this checkpoint")
    print("  - Call clear_checkpoint() to start fresh")
    print("=" * 60)


class CheckpointManager:
    """Manages checkpoint files for ML strategy training."""
    
    def __init__(self, checkpoint_dir: str = 'checkpoints'):
        """Initialize checkpoint manager.
        
        Args:
            checkpoint_dir: Directory path for storing checkpoint files
        """
        self.checkpoint_path = Path(checkpoint_dir)
        self.checkpoint_path.mkdir(exist_ok=True)
        # Use datetime stamp for new checkpoint files
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.checkpoint_file = self.checkpoint_path / f'ml_strategies_checkpoint_{timestamp}.pkl'
        self.best_params_file = self.checkpoint_path / f'ml_best_params_{timestamp}.pkl'
    
    def load_checkpoint(self, ml_models: Dict[str, Tuple[Any, str]]) -> Tuple[List, Set[str]]:
        """Load checkpoint with completed strategies and model names.
        
        NOTE: Always returns empty results to force training new models each run.
        Old checkpoints are preserved with timestamps.
        
        Args:
            ml_models: Dictionary of model_name -> (model_instance, short_name) pairs
                      Used to validate checkpoint against current configuration
        
        Returns:
            Tuple of (empty ml_strategies list, empty completed_models set)
        """
        ml_strategies = []
        completed_models = set()
        
        # Always start fresh - never load old checkpoints
        print("\n✓ Starting fresh training run (checkpoints disabled)")
        
        return ml_strategies, completed_models
    
    def save_checkpoint(self, ml_strategies: List, completed_models: Set[str]) -> None:
        """Save checkpoint with current progress.
        
        Args:
            ml_strategies: List of completed MLStrategy instances
            completed_models: Set of model names that have been completed
        """
        try:
            checkpoint_data = {
                'strategies': ml_strategies,
                'completed_models': completed_models,
                'timestamp': pd.Timestamp.now()
            }
            with open(self.checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint_data, f)
        except Exception as e:
            raise RuntimeError(f"Could not save checkpoint: {e}")
    
    def remove_checkpoint(self) -> None:
        """Remove checkpoint file after all models complete."""
        try:
            if self.checkpoint_file.exists():
                self.checkpoint_file.unlink()
                print(f"\n✓ All models complete! Checkpoint file removed.")
        except Exception as e:
            print(f"Warning: Could not remove checkpoint file: {e}")
    
    def load_best_params(self) -> Dict[str, Dict]:
        """Load saved best hyperparameters from previous runs.
        
        NOTE: Always returns empty dict to force hyperparameter tuning each run.
        Old parameters are preserved with timestamps.
        
        Returns:
            Empty dictionary (always trains with fresh hyperparameter search)
        """
        saved_best_params = {}
        
        # Always start fresh - never load old parameters
        print("✓ Hyperparameter tuning will run for all models (saved params disabled)")
        
        return saved_best_params
    
    def save_best_params(self, best_params: Dict[str, Dict]) -> None:
        """Save best hyperparameters for future runs.
        
        Args:
            best_params: Dictionary mapping model_name -> best_params dict
        """
        try:
            with open(self.best_params_file, 'wb') as f:
                pickle.dump(best_params, f)
        except Exception as e:
            raise RuntimeError(f"Could not save best parameters: {e}")
    
    def checkpoint_exists(self) -> bool:
        """Check if checkpoint file exists."""
        return self.checkpoint_file.exists()
    
    def best_params_exists(self) -> bool:
        """Check if best params file exists."""
        return self.best_params_file.exists()

In [8]:
"""
helper functions for ML dataset building and feature selection
"""

def lagged_returns(prices: pd.Series, lags: List[int]) -> pd.DataFrame:
    """Return DataFrame of lagged pct-change returns for given lags (in days).

    Each column is named 'ret_{lag}'.
    """
    out = pd.DataFrame(index=prices.index)
    for lag in lags:
        out[f'ret_{lag}'] = prices.pct_change(lag).shift(1)
    return out


def moving_averages(prices: pd.Series, windows: List[int]) -> pd.DataFrame:
    out = pd.DataFrame(index=prices.index)
    for w in windows:
        out[f'sma_{w}'] = prices.rolling(window=w, min_periods=int(w * 0.8)).mean()
        out[f'ema_{w}'] = prices.ewm(span=w, adjust=False).mean()
    return out


def build_ml_dataset(prices: pd.Series, exog: pd.DataFrame = None, lags=[5,15,30,60], ma_windows=[21,63,252], forward=5):
    """Build a simple supervised dataset for predicting forward `forward`-day return.

    Returns X, y aligned with index dropped where NaN.
    """
    lagged = lagged_returns(prices, lags)
    mas = moving_averages(prices, ma_windows)

    X = pd.concat([lagged, mas], axis=1)
    if exog is not None:
        X = pd.concat([X, exog.shift(1)], axis=1)

    y = prices.pct_change(forward).shift(-forward)

    # drop rows with NaN
    df = pd.concat([X, y.rename('target')], axis=1).dropna()

    X_clean = df.drop(columns=['target'])
    y_clean = df['target']
    return X_clean, y_clean


def filter_features_by_ic(X: pd.DataFrame, y: pd.Series, test_start_date, 
                          ic_percentile: float, correlation_threshold: float = 0.8) -> Tuple[pd.DataFrame, pd.Series]:
    """Filter features by Sharpe ratio using two-stage filtering (train-only selection).

    Stage 1: Rank features by the absolute Sharpe ratio of a strategy that uses
             positions based on feature sign: positions = np.where(feature > 0, 1, -1),
             then calculates strategy_returns = y_train * positions.shift(1).
             This matches the actual strategy logic (see BaseStrategy.calculate_returns).
             Keep the top ``ic_percentile`` percent by Sharpe.

    Stage 2: Remove highly correlated feature pairs (based on training data
             correlations), keeping the one with higher Stage 1 score.

    Prevents look-ahead bias by calculating Sharpe and correlations only on
    training data, then applying the same feature selection to the full dataset.

    Args:
        X: Feature DataFrame with datetime index
        y: Target Series with datetime index (e.g., next-period log returns)
        test_start_date: Date that splits train/test periods
        ic_percentile: Top percentile of features to keep (0-100)
        correlation_threshold: Threshold for removing correlated features (default 0.8)

    Returns:
        Tuple of (X_filtered, y_filtered) with aligned indices after dropping NaN rows
    """
    if test_start_date is None:
        print("Warning: Sharpe filtering requires test_start_date to prevent look-ahead bias. Skipping filtering.")
        return X, y
    
    # Calculate metrics using ONLY training data
    train_mask = X.index < test_start_date
    X_train = X[train_mask]
    y_train = y[train_mask]
    
    if X_train.empty or y_train.empty:
        print("Warning: No training data available for Sharpe filtering. Skipping filtering.")
        return X, y
    
    # Remove columns with more than threshold% NaN values before filtering
    nan_threshold = ML_PARAMS.get('nan_threshold', 0.3)
    nan_percentages = X.isna().mean()
    cols_to_drop = nan_percentages[nan_percentages > nan_threshold].index.tolist()
    
    if cols_to_drop:
        print(f"Pre-filtering: Removing {len(cols_to_drop)} columns with >{nan_threshold*100}% NaN values")
        X = X.drop(columns=cols_to_drop)
        X_train = X[train_mask]
    
    # STAGE 1: Calculate Sharpe ratio for each feature using actual strategy logic
    # Match the train period calculation: passive_returns * positions.shift(1)
    # where positions = np.where(feature > 0, 1, -1) to replicate Coincident strategy logic
    # 
    # IMPORTANT: Features like `TLT_signal` are ALREADY shifted in strategy generation,
    # so we should NOT shift positions again. The feature value at time t uses data up to t-1.
    # Strategy logic: positions[t] = where(signal[t] > 0, 1, -1)
    #                 strategy_returns[t] = passive_returns[t] * positions[t-1]
    # Since signal is already lagged, positions[t] uses info up to t-1
    # Then positions.shift(1) gives us positions[t-1] for time t
    # 
    # For features that are pre-shifted signals: positions = where(feature > 0, 1, -1)
    # Then: strategy_returns = y_train * positions.shift(1)
    sharpe_values = {}
    sharpe_values_signed = {}
    for col in X_train.columns:
        # Generate positions from feature: +1 when feature > 0, -1 otherwise
        positions = pd.Series(np.where(X_train[col] > 0, 1, -1), index=X_train.index)
        # Calculate strategy returns using same logic as BaseStrategy.calculate_returns()
        # strategy_returns = passive_returns * positions.shift(1)
        strat_ret = (y_train.shift(1) * positions.shift(1)).fillna(0)
        
        if strat_ret.std(ddof=1) == 0 or strat_ret.empty:
            sharpe = 0.0
        else:
            sharpe = float(PerformanceMetrics.calculate_sharpe_ratio(strat_ret, periods_per_year=252))
            
        # sharpe_values_signed[col] = sharpe
        sharpe_values[col] = abs(sharpe)

    # Calculate the threshold for top percentile
    sharpe_series = pd.Series(sharpe_values)
    
    # Print all Sharpe values (disable truncation)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print("\nAll Feature Sharpe Ratios (sorted descending):")
        print(sharpe_series.sort_values(ascending=False))
    
    threshold = sharpe_series.quantile((100 - ic_percentile) / 100)

    # Select features above the threshold
    stage1_sorted = sharpe_series[sharpe_series >= threshold].sort_values(ascending=False, kind='mergesort')
    stage1_features = stage1_sorted.index.tolist()
    
    if len(stage1_features) == 0:
        print(f"Warning: IC filtering with percentile={ic_percentile} removed all features. Using all features.")
        return X, y

    
    # STAGE 2: Remove highly correlated features, keeping the one with higher IC
    # Calculate correlation matrix using ONLY training data
    X_train_stage1 = X_train[stage1_features]
    corr_matrix = X_train_stage1.corr().abs()
    
    # Track features to keep
    features_to_keep = []
    features_removed = []
    
    # Iterate through features sorted by Sharpe (highest first)
    for feature in stage1_features:
        # Check if this feature is already removed due to correlation with a higher-IC feature
        if feature in features_removed:
            continue
        
        # Keep this feature
        features_to_keep.append(feature)
        
        # Find and remove features highly correlated with this one
        for other_feature in stage1_features:
            if other_feature == feature or other_feature in features_removed:
                continue
            
            # If highly correlated, remove the one with lower Sharpe (which is the current other_feature)
            corr_val = corr_matrix.loc[feature, other_feature]
            if pd.notna(corr_val) and corr_val >= correlation_threshold:
                features_removed.append(other_feature)
    
    if len(features_to_keep) == 0:
        print(f"Warning: Correlation filtering removed all features. Using Stage 1 features.")
        features_to_keep = stage1_features

    # Apply same feature selection to entire dataset (train + test)
    X_filtered = X[features_to_keep]
    
    # Drop rows with any remaining NaN values from the entire dataset
    rows_before_total = len(X_filtered)
    X_filtered = X_filtered.dropna()
    rows_dropped_total = rows_before_total - len(X_filtered)
    if rows_dropped_total > 0:
        print(f"Final cleaning: Dropped {rows_dropped_total} rows with NaN values from full dataset ({rows_dropped_total/rows_before_total*100:.1f}%)")
    
    print(f"Stage 2 - Correlation filtering: removed {len(features_removed)} highly correlated features (threshold={correlation_threshold})")
    print(f"  Final feature count: {len(features_to_keep)}/{len(stage1_features)}")
    if features_removed:
        print(f"  Removed features: {features_removed[:10]}{'...' if len(features_removed) > 10 else ''}")
    
    # Align y with the cleaned X_filtered (same rows)
    y_filtered = y.loc[X_filtered.index]
    
    return X_filtered, y_filtered


def build_ml_features_from_strategies(prices: pd.DataFrame, strategies: list, 
                                      test_start_date=None) -> Tuple[pd.DataFrame, pd.Series]:
    """Aggregate numeric indicator columns from each strategy into a feature DataFrame.

    Args:
        prices: Price DataFrame
        strategies: List of rule-based strategies to extract features from
        test_start_date: Optional date to split train/test for IC filtering (prevents look-ahead bias)

    Returns:
        Tuple of (X, y) where X is feature DataFrame and y is Series of forward log returns.
    """
    feature_frames = []

    # Ensure strategies have indicator columns populated
    for strat in strategies:
        # compute indicators/returns if not yet present
        try:
            strat.calculate_returns()
        except Exception:
            # ignore if already computed or not applicable
            pass

        df = strat.data.copy()
        # Exclude: returns, positions, trades, Close (redundant with price)
        # Also exclude raw SMAs/EMAs (they're trending - only keep normalized spreads)
        # Also exclude price column from each strategy (trending, use price_zscore instead)
        # Also exclude individual window returns (redundant across strategies - keep only signals)
        exclude = {
            "strategy_returns", "cum_strategy_returns", "passive_returns", 
            "cum_passive_returns", "position", "trades", "Close", "positions"
        }
        
        # Get the price column name (first column in prices DataFrame)
        price_col = prices.columns[0] if len(prices.columns) > 0 else 'price'
        
        indicator_cols = []
        for c in df.select_dtypes(include=["number"]).columns:
            if c in exclude:
                continue
            # Exclude price column (trending, we'll use price_zscore instead)
            if c == price_col:
                continue
            # Exclude correlation/weight diagnostic columns from multi-coincident strategies
            if c.endswith('_correlation') or c.endswith('_weight'):
                continue
            # Exclude raw SMA/EMA columns (keep only normalized spreads/z-scores)
            if c.startswith('sma_') or c.startswith('ema_'):
                continue
            # Exclude raw coincident index prices (they're trending)
            # Keep _price_zscore columns (already normalized in strategy)
            if c.endswith('_price'):
                continue
            # Exclude individual window returns (redundant - keep only aggregated signals)
            # These returns appear multiple times across different multi-window strategies
            if c.startswith('ret_') or c.endswith('_ret_5d') or c.endswith('_ret_10d') or \
               c.endswith('_ret_20d') or c.endswith('_ret_60d'):
                continue
            indicator_cols.append(c)
        
        if indicator_cols:
            tmp = df[indicator_cols].copy()
            
            # prefix columns with strategy name to avoid collisions
            tmp.columns = [f"{strat.name}__{c}" for c in tmp.columns]
            feature_frames.append(tmp)

    if not feature_frames:
        return pd.DataFrame(index=prices.index), pd.Series(dtype=float)

    # Add normalized price (z-score) instead of raw price
    price_series = prices[prices.columns[0]]
    price_zscore = zscore(price_series, window=ZSCORE_WINDOW)
    price_df = pd.DataFrame({'price_zscore': price_zscore}, index=prices.index)
    
    X = pd.concat(feature_frames + [price_df], axis=1).sort_index()
    print(f"Built ML feature set with {X.shape[1]} features from {len(strategies)} strategies.")
    print(f"Feature names: {X.columns.tolist()}")

    # compute forward return as target (using original price for returns calculation)
    log_ret = np.log(price_series / price_series.shift(1))
    y = log_ret.shift(-1).rename('fwd_log_ret')

    # drop rows with NaNs in features or target
    combined = pd.concat([X, y], axis=1)
    X_clean = combined.drop(columns=['fwd_log_ret'])
    y_clean = combined['fwd_log_ret']
    
    # Filter features by Information Coefficient (IC) if percentile is specified
    # IMPORTANT: IC filtering uses ONLY training data to prevent look-ahead bias
    ic_percentile = ML_PARAMS.get('ic_percentile')
    correlation_threshold = ML_PARAMS.get('correlation_threshold', 0.8)
    if ic_percentile is not None and 0 < ic_percentile <= 100:
        X_clean, y_clean = filter_features_by_ic(X_clean, y_clean, test_start_date, ic_percentile, correlation_threshold)
    
    return X_clean, y_clean


In [9]:
"""
ML-based Strategy
"""

class MLStrategy:
    """ML-based trading strategy that works with any sklearn-compatible model.
    
    Builds features from rule-based strategies, trains a regressor to predict forward returns,
    and creates long/short positions based on predictions.
    """

    def __init__(self, prices: pd.DataFrame, feature_strategies: list, test_start_date, 
                 model: Any, model_name: str = "ML", threshold: float = 0.0,
                 prebuilt_features: tuple = None):
        """
        Args:
            prices: Price DataFrame
            feature_strategies: List of rule-based strategies to extract features from
            test_start_date: Date to split train/test
            model: sklearn-compatible model instance
            model_name: Short name for the model (e.g., 'RF', 'LinReg')
            threshold: Prediction threshold for long/short signals (default 0.0)
            prebuilt_features: Optional tuple of (X, y) pre-computed features to avoid rebuilding
        """
        self.prices = prices
        self.feature_strategies = feature_strategies
        self.test_start_date = test_start_date
        self.model = model
        self.model_name = model_name
        self.name = model_name
        self.threshold = threshold
        self.data = None
        self._trained = False
        self.train_mse = None
        self.test_mse = None
        self.prebuilt_features = prebuilt_features
        
    def fit(self, X: pd.DataFrame, y: pd.Series):
        """Train the underlying model."""
        self.model.fit(X, y)
        self._trained = True

    def predict(self, X: pd.DataFrame) -> pd.Series:
        """Predict forward returns using the trained model."""
        if not self._trained:
            raise RuntimeError("Model not fitted. Call calculate_returns() first.")
        return pd.Series(self.model.predict(X), index=X.index)

    def signals_from_preds(self, preds: pd.Series) -> pd.Series:
        """Convert predictions to long/short signals."""
        signal = pd.Series(np.where(preds > self.threshold, 1, -1), index=preds.index)
        return signal
        
    def calculate_returns(self):
        """Build features, train on train period, generate signals/returns for full period."""
        if self.data is not None:
            return  # already computed
            
        # Use prebuilt features if available, otherwise build from strategies
        if self.prebuilt_features is not None:
            X, y = self.prebuilt_features
        else:
            X, y = build_ml_features_from_strategies(self.prices, self.feature_strategies, self.test_start_date)
        
        if X.empty or y.empty:
            print(f"Warning: No ML features available for {self.name}; creating empty data.")
            self.data = pd.DataFrame(index=self.prices.index)
            return
        
        # Update name with feature count
        self.name = f"{self.model_name}_{len(X.columns)}f"
        
        # split by time index
        X_train = X.loc[X.index < self.test_start_date]
        y_train = y.loc[y.index < self.test_start_date]
        
        if X_train.empty:
            print(f"Warning: Insufficient training data for {self.name}; creating empty data.")
            self.data = pd.DataFrame(index=self.prices.index)
            return
        
        # Train on training period
        self.fit(X_train, y_train)
        
        # Predict on full period where features exist
        preds = self.predict(X)
        
        # Calculate MSE for train and test sets
        X_test = X.loc[X.index >= self.test_start_date]
        y_test = y.loc[y.index >= self.test_start_date]
        
        train_preds = self.predict(X_train)
        test_preds = self.predict(X_test)
        
        self.train_mse = ((y_train - train_preds) ** 2).mean()
        self.test_mse = ((y_test - test_preds) ** 2).mean()
        
        signals = self.signals_from_preds(preds)
        
        # Build data DataFrame matching other strategies
        price_col = self.prices.columns[0]
        self.data = pd.DataFrame({price_col: self.prices[price_col].reindex(X.index)})
        self.data['passive_returns'] = np.log(self.data[price_col] / self.data[price_col].shift(1)).fillna(0)
        
        # Standardize on 'positions' (plural) across all strategies
        self.data['positions'] = signals.reindex(self.data.index).fillna(0)
        self.data['strategy_returns'] = self.data['positions'].shift(1).fillna(0) * self.data['passive_returns']
        # Cumulative returns: cumulative product of exp(log_returns)
        self.data['cum_passive_returns'] = self.data['passive_returns'].cumsum().apply(np.exp)
        self.data['cum_strategy_returns'] = self.data['strategy_returns'].cumsum().apply(np.exp)
        self.data['trades'] = self.data['positions'].diff().abs().fillna(0)


In [10]:



def create_ml_models(tune_hyperparameters=False, saved_best_params=None):
    """Create a dictionary of ML models for strategy testing.
    
    Args:
        tune_hyperparameters: If True, returns GridSearchCV wrapped models for tuning.
                             If False, returns models with default parameters.
        saved_best_params: Dict mapping model names to their best parameters from previous runs.
                          If provided, these params will be used instead of tuning.
    
    Returns a dict with model_name -> (model_instance, short_name) pairs.
    """
    if saved_best_params is None:
        saved_best_params = {}
    
    random_seed = ML_PARAMS.get('random_seed', 42)
    
    base_models = {
        'Linear Regression': (LinearRegression(), 'LinReg'),
        'Elastic Net': (ElasticNet(random_state=random_seed), 'ElasticNet'),
        'LASSO': (Lasso(random_state=random_seed), 'LASSO'),
        'XGBoost' : (XGBRegressor(random_state=random_seed, n_jobs=ML_PARAMS.get('n_jobs', -1)), 'XGB'),
        # 'Support Vector Machine': (SVR(), 'SVM'),
        'K-Nearest Neighbor': (KNeighborsRegressor(), 'KNN'),
        'Decision Tree': (DecisionTreeRegressor(random_state=random_seed), 'DTree'),
        'Extra Trees': (ExtraTreesRegressor(random_state=random_seed), 'ExtraTrees'),
        'Random Forest': (RandomForestRegressor(random_state=random_seed), 'RF'),
        'Gradient Boosting': (GradientBoostingRegressor(random_state=random_seed), 'GBT'),
        'Adaptive Boosting': (AdaBoostRegressor(random_state=random_seed), 'AdaBoost'),
    }
    
    tuned_models = {}
    
    for model_name, (model, short_name) in base_models.items():
        # Check if we have saved best params for this model
        if model_name in saved_best_params:
            print(f"  → Using saved best params for {model_name}: {saved_best_params[model_name]}")
            model.set_params(**saved_best_params[model_name])
            tuned_models[model_name] = (model, short_name)
        elif tune_hyperparameters:
            # Only tune if we don't have saved params
            param_grid = ML_HYPERPARAMETER_GRIDS.get(model_name, {})
            
            if param_grid:
                # Use GridSearchCV with 3-fold CV, optimize for negative MSE
                grid_search = GridSearchCV(
                    estimator=model,
                    param_grid=param_grid,
                    cv=3,
                    scoring='neg_mean_squared_error',
                    n_jobs=ML_PARAMS.get('n_jobs', -1),
                    verbose=0
                )
                tuned_models[model_name] = (grid_search, short_name)
            else:
                # No hyperparameters to tune (e.g., LinearRegression)
                tuned_models[model_name] = (model, short_name)
        else:
            # No tuning, just use default params
            tuned_models[model_name] = (model, short_name)
    
    return tuned_models



In [11]:
"""

"""
def create_rule_based_strategies(prices: pd.DataFrame) -> List:
    """Create all rule-based (non-ML) trading strategies.
    
    Args:
        prices: Price DataFrame
    
    Returns:
        List of strategy instances
    """
    strategies = []
    
    # Momentum strategies
    for short, long in MOMENTUM_PARAMS['sma_short_long_pairs']:
        strategies.append(SMAStrategy(prices, short, long))

    for fast, slow, signal in MOMENTUM_PARAMS['macd_params']:
        strategies.append(MACDStrategy(prices, fast, slow, signal))

    # Mean reversion strategies
    strategies.append(ZScoreStrategy(
        prices,
        MEAN_REVERSION_PARAMS['zscore_window'],
        MEAN_REVERSION_PARAMS['zscore_threshold']
    ))

    rsi_variants = generate_rsi_variants(
        prices,
        periods=MEAN_REVERSION_PARAMS.get('rsi_period'),
        threshold_pairs=MEAN_REVERSION_PARAMS.get('rsi_threshhold_pairs')
    )
    strategies.extend(rsi_variants)
    
    bollinger_variants = generate_bollinger_variants(
        prices,
        windows=MEAN_REVERSION_PARAMS.get('bollinger_windows'),
        std_devs=MEAN_REVERSION_PARAMS.get('bollinger_std_devs')
    )
    strategies.extend(bollinger_variants)
    
    # Volume and oscillator strategies
    if VOLUME_OSCILLATOR_PARAMS.get('enabled', False):
        for ticker, window in VOLUME_OSCILLATOR_PARAMS.get('obv', []):
            try:
                strategies.append(OBVStrategy(prices, ticker, window))
            except Exception as e:
                print(f"Warning: Could not create OBVStrategy for {ticker}: {e}")
        
        for ticker, k_period, d_period, oversold, overbought in VOLUME_OSCILLATOR_PARAMS.get('stochastic', []):
            try:
                strategies.append(StochasticStrategy(
                    prices, ticker, k_period, d_period, oversold, overbought
                ))
            except Exception as e:
                print(f"Warning: Could not create StochasticStrategy for {ticker}: {e}")
        
        for period, threshold in VOLUME_OSCILLATOR_PARAMS.get('roc', []):
            try:
                strategies.append(ROCStrategy(prices, period, threshold))
            except Exception as e:
                print(f"Warning: Could not create ROCStrategy: {e}")
    
    # Coincident indices strategies
    if COINCIDENT_INDICES_PARAMS.get('enabled', False):
        for ticker, window in COINCIDENT_INDICES_PARAMS.get('single_indices', []):
            try:
                strategies.append(CoincidentIndexStrategy(
                    prices, 
                    coincident_ticker=ticker,
                    window=window,
                    correlation_threshold=COINCIDENT_INDICES_PARAMS.get('correlation_threshold', 0.0)
                ))
            except Exception as e:
                print(f"Warning: Could not create CoincidentIndexStrategy for {ticker}: {e}")
        
        for tickers, window, aggregation in COINCIDENT_INDICES_PARAMS.get('multi_indices', []):
            try:
                strategies.append(MultiCoincidentStrategy(
                    prices,
                    coincident_tickers=tickers,
                    window=window,
                    aggregation=aggregation,
                    include_individual_features=False,
                    correlation_window=MULTI_WINDOW_PARAMS.get('correlation_window', 60)
                ))
            except Exception as e:
                print(f"Warning: Could not create MultiCoincidentStrategy: {e}")
    
    # Multi-window returns strategies
    if MULTI_WINDOW_PARAMS.get('enabled', False):
        for windows, signal_method in MULTI_WINDOW_PARAMS.get('same_asset_strategies', []):
            try:
                strategies.append(MultiWindowReturnsStrategy(
                    prices,
                    windows=windows,
                    signal_method=signal_method
                ))
            except Exception as e:
                print(f"Warning: Could not create MultiWindowReturnsStrategy: {e}")
        
        for ref_ticker, windows, signal_method in MULTI_WINDOW_PARAMS.get('cross_asset_strategies', []):
            try:
                strategies.append(CrossAssetWindowReturnsStrategy(
                    prices,
                    reference_ticker=ref_ticker,
                    windows=windows,
                    signal_method=signal_method
                ))
            except Exception as e:
                print(f"Warning: Could not create CrossAssetWindowReturnsStrategy for {ref_ticker}: {e}")
    
    return strategies


def create_ml_strategies(prices: pd.DataFrame, rule_based_strategies: List, 
                        test_start_date, tune_hyperparameters: bool = True,
                        checkpoint_dir: str = 'checkpoints') -> List:
    """Create ML-based trading strategies using rule-based features.
    
    NOTE: Always trains new models from scratch. Checkpoints are saved with 
    datetime stamps for historical reference but never loaded.
    
    Args:
        prices: Price DataFrame
        rule_based_strategies: List of rule-based strategies to extract features from
        test_start_date: Date to split train/test
        tune_hyperparameters: If True, use GridSearchCV to tune hyperparameters
        checkpoint_dir: Directory for saving checkpoints (with datetime stamps)
    
    Returns:
        List of MLStrategy instances
    """
    checkpoint_mgr = CheckpointManager(checkpoint_dir)
    
    # Never load saved params - always train fresh
    saved_best_params = {}
    
    # Create ML models
    ml_models = create_ml_models(
        tune_hyperparameters=tune_hyperparameters,
        saved_best_params=saved_best_params
    )
    
    # Load checkpoint (will always be empty - just for consistency)
    ml_strategies, completed_models = checkpoint_mgr.load_checkpoint(ml_models)
    
    # Build features once for all ML strategies (optimization)
    print("\nBuilding ML features from rule-based strategies...")
    X, y = build_ml_features_from_strategies(prices, rule_based_strategies, test_start_date)
    if X.empty or y.empty:
        print("Warning: No ML features could be built; skipping all ML strategies.")
        return []
    print(f"✓ Built {X.shape[1]} features from {len(rule_based_strategies)} strategies")
    print(f"  Feature shape: {X.shape}, Target shape: {y.shape}")
    
    # Train all models (fresh each run)
    tuning_msg = " with hyperparameter tuning" if tune_hyperparameters else ""
    print(f"\nCreating ML strategies{tuning_msg}...")
    total_models = len(ml_models)
    
    # Track starting count to calculate correct progress numbers
    initial_completed = len(completed_models)
    
    for idx, (model_name, (model_instance, short_name)) in enumerate(ml_models.items(), 1):
        try:
            current_progress = initial_completed + idx
            print(f"\n[{current_progress}/{total_models}] Training {model_name}...")
            
            ml_strat = MLStrategy(
                prices=prices,
                feature_strategies=rule_based_strategies,
                test_start_date=test_start_date,
                model=model_instance,
                model_name=short_name,
                prebuilt_features=(X, y)
            )
            
            ml_strat.calculate_returns()
            
            if ml_strat.data is not None and not ml_strat.data.empty:
                ml_strategies.append(ml_strat)
                completed_models.add(model_name)
                
                # Extract and save best params if tuned
                if hasattr(ml_strat.model, 'best_params_'):
                    saved_best_params[model_name] = ml_strat.model.best_params_
                    print(f"✓ {ml_strat.name}: best_params={ml_strat.model.best_params_}")
                else:
                    print(f"✓ {ml_strat.name}")
                
                # Save checkpoint after each successful model (with datetime stamp)
                try:
                    checkpoint_mgr.save_checkpoint(ml_strategies, completed_models)
                    if saved_best_params:
                        checkpoint_mgr.save_best_params(saved_best_params)
                    print(f"  → Checkpoint saved with timestamp ({len(completed_models)}/{total_models} complete)")
                except Exception as e:
                    print(f"  Warning: Could not save checkpoint: {e}")
            else:
                print(f"✗ {model_name} produced empty data; skipping.")
        except Exception as e:
            print(f"✗ Could not create {model_name} strategy: {e}")
            print(f"  → Progress saved. You can resume by running again.")
    
    # Don't cleanup checkpoint - keep it with timestamp for historical reference
    if len(completed_models) == total_models:
        print(f"\n✓ All models complete!")
        print(f"✓ Best hyperparameters saved to: {checkpoint_mgr.best_params_file}")
        print(f"✓ Checkpoint saved to: {checkpoint_mgr.checkpoint_file}")
    
    return ml_strategies


def init_strategies(prices: pd.DataFrame, test_start_date=None) -> List:
    """Create all trading strategy instances.
    
    Args:
        prices: Price DataFrame
        test_start_date: Optional date to split train/test for ML strategies
    
    Returns:
        List of all strategy instances (rule-based + ML if test_start_date provided)
    """
    # Create rule-based strategies
    strategies = create_rule_based_strategies(prices)
    
    # Add ML strategies if test_start_date provided
    if test_start_date is not None:
        tune_hyperparams = ML_PARAMS.get('tune_hyperparameters', True)
        ml_strats = create_ml_strategies(prices, strategies, test_start_date, tune_hyperparams)
        strategies.extend(ml_strats)
    
    return strategies


__all__ = ["create_rule_based_strategies", "create_ml_strategies", "init_strategies"]


In [12]:
class BacktestEngine:
    """Execute backtests with realistic commission costs."""
    
    def __init__(self, initial_capital=100_000, commission=5):
        self.initial_capital = initial_capital
        self.commission = commission
    
    def run_backtest(self, strategy_data):
        """Run backtest with commission costs."""
        capital = self.initial_capital
        portfolio_values = []
        
        for idx, row in strategy_data.iterrows():
            # Apply returns
            capital *= np.exp(row['strategy_returns'])
            
            # Deduct commission on trades
            if row.get('trades', 0) != 0:
                capital -= self.commission
            
            portfolio_values.append(capital)
        
        strategy_data['portfolio_value'] = portfolio_values
        strategy_data['portfolio_returns'] = (
            pd.Series(portfolio_values, index=strategy_data.index).pct_change()
        )
        
        final_value = portfolio_values[-1]
        total_return = (final_value / self.initial_capital - 1) * 100
        
        return strategy_data, final_value, total_return

In [ ]:
"""
Backtest
"""
def compute_spy_benchmark(test_start_date, align_indices: Optional[Iterable[pd.Index]] = None, period: str = 'test') -> Optional[Dict]:
    """Fetch SPY and compute passive buy-and-hold metrics for the requested period.

    Args:
        test_start_date: Timestamp splitting train/test
        align_indices: Optional iterable of indices to align SPY to the common
            intersection across provided indices for apples-to-apples metrics.
        period: 'test' (>= test_start_date) or 'train' (< test_start_date)

    Returns:
        Dict with 'data' and 'metrics' keys, or None on failure
    """
    try:
        spy_loader = DataLoader('SPY', START_DATE, END_DATE)
        spy_prices = spy_loader.get_prices()
        if isinstance(spy_prices, pd.Series):
            spy_prices = spy_prices.to_frame()
    except Exception as e:
        print(f"Warning: could not load SPY benchmark ({e})")
        return None

    try:
        if period == 'train':
            spy_window = spy_prices.loc[spy_prices.index < test_start_date].copy()
            period_name = 'train'
        else:
            spy_window = spy_prices.loc[spy_prices.index >= test_start_date].copy()
            period_name = 'test'
        if spy_window.empty:
            print(f"Warning: SPY has no data in {period_name} period; benchmark not added.")
            return None

        price_col = spy_window.columns[0]
        # Compute log returns (same convention as strategies)
        spy_window['passive_returns'] = (spy_window[price_col] / spy_window[price_col].shift(1)).apply(
            lambda x: np.log(x) if x > 0 else 0
        )
        spy_window['passive_returns'] = spy_window['passive_returns'].fillna(0)
        
        # IMPORTANT: Align with strategy convention where positions are shifted by 1 day
        # The first test-period strategy return is 0 due to the 1-day position shift.
        # To compare apples-to-apples, set SPY's first test-period return to 0 as well.
        if not spy_window['passive_returns'].empty:
            spy_window.iloc[0, spy_window.columns.get_loc('passive_returns')] = 0.0

        # Optional alignment to strategy indices (common intersection)
        if align_indices:
            try:
                iter_indices = list(align_indices)
                if len(iter_indices) > 0:
                    common_idx = iter_indices[0]
                    for idx in iter_indices[1:]:
                        common_idx = common_idx.intersection(idx)
                    # Slice SPY to the common index intersection
                    aligned = spy_window.loc[spy_window.index.intersection(common_idx)].copy()
                    if not aligned.empty:
                        # Reset first aligned day's passive return to 0 and recompute cumsum base
                        first_idx = aligned.index[0]
                        aligned.iloc[0, aligned.columns.get_loc('passive_returns')] = 0.0
                        spy_window = aligned
            except Exception as e:
                print(f"Warning: Could not align SPY to provided indices: {e}")

        spy_window['cum_passive_returns'] = spy_window['passive_returns'].cumsum().apply(np.exp)

        spy_metrics = PerformanceMetrics.calculate_all_metrics(
            spy_window['passive_returns'],
            spy_window['cum_passive_returns']
        )
        spy_metrics['Final Value'] = spy_window['cum_passive_returns'].iloc[-1] * INITIAL_CAPITAL

        return {
            'data': spy_window,
            'metrics': spy_metrics
        }
    except Exception as e:
        print(f"Warning computing SPY benchmark metrics: {e}")
        return None


def save_and_print_results(results: Dict) -> pd.DataFrame:
    """Build comparison DataFrame from results, print and save to CSV.
    
    Args:
        results: Dictionary mapping strategy_name -> result dict with 'metrics' key
    
    Returns:
        Comparison DataFrame sorted by Total Return (descending)
    """
    comparison_df = pd.DataFrame({name: res['metrics'] for name, res in results.items()}).T
    
    # Sort by Total Return in descending order
    comparison_df = comparison_df.sort_values('Total Return', ascending=False)

    print("\n" + "="*80)
    print("STRATEGY COMPARISON")
    print("="*80)
    print(comparison_df.to_string())

    comparison_df.to_csv('strategy_comparison.csv')
    print("\nResults saved to strategy_comparison.csv")
    return comparison_df


def save_detailed_results(results: Dict, out_dir: str = "results") -> None:
    """Save per-strategy backtest DataFrames and a summary CSV for diagnostics.
    
    Args:
        results: Dictionary mapping strategy_name -> result dict
        out_dir: Output directory for results
    
    Writes:
        - results/strategy_data/<strategy>.csv for each strategy
        - results/summary.csv with aggregated metrics
    """
    out_path = Path(out_dir)
    data_dir = out_path / "strategy_data"
    out_path.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)

    def safe_name(name: str) -> str:
        """Convert strategy name to safe filename."""
        return re.sub(r"[^A-Za-z0-9._\-]+", "_", name)[:200]

    summary_rows = []
    for name, res in results.items():
        bt_data = res.get('data')
        final_value = res.get('final_value')
        total_return = res.get('total_return')
        metrics = res.get('metrics', {}) or {}

        # Save per-strategy data if available
        if isinstance(bt_data, pd.DataFrame) and not bt_data.empty:
            file_path = data_dir / f"{safe_name(name)}.csv"
            try:
                bt_data.to_csv(file_path, index_label='date')
            except Exception as e:
                print(f"Warning: Failed to save data for {name}: {e}")

        # Build summary row
        row = {
            'strategy': name,
            'final_value': final_value,
            'total_return_pct': total_return,
        }
        # Merge metrics (ensure flat dict)
        if isinstance(metrics, dict):
            row.update(metrics)
        summary_rows.append(row)

    # Save summary CSV
    try:
        summary_df = pd.DataFrame(summary_rows)
        summary_csv_path = out_path / 'summary.csv'
        summary_df.sort_values('total_return_pct', ascending=False).to_csv(summary_csv_path, index=False)
        print(f"Detailed results saved to: {out_path.resolve()} (data + summary.csv)")

        # Optional: Create Sharpe alignment debug CSV if Stage 1 diagnostics exist
        try:
            stage1_csv = Path(__file__).resolve().parents[1] / 'results' / 'debug_feature_engineering_sharpes.csv'
            if stage1_csv.exists():
                sharpe_df = pd.read_csv(stage1_csv)
                # Choose best feature per strategy by abs Sharpe
                idx_best = sharpe_df.groupby('strategy')['sharpe_stage1_abs'].idxmax()
                best_view = sharpe_df.loc[idx_best].reset_index(drop=True).rename(
                    columns={'feature': 'best_feature'}
                )

                # Merge with summary Sharpe Ratio as fallback if train/test comparison isn't available
                res_df = summary_df.copy()
                if 'strategy' not in res_df.columns:
                    # Attempt to find strategy column
                    if res_df.columns[0].lower() in ('strategy', 'name', 'strategy_name'):
                        res_df = res_df.rename(columns={res_df.columns[0]: 'strategy'})
                keep_cols = ['strategy'] + [c for c in res_df.columns if 'sharpe' in c.lower()]
                res_sel = res_df[keep_cols]
                merged = best_view.merge(res_sel, on='strategy', how='left')
                out_path_align = out_path / 'debug_sharpe_alignment.csv'
                merged.to_csv(out_path_align, index=False)
                print(f"Sharpe alignment (with summary.csv) saved to: {out_path_align}")
            else:
                print("Sharpe alignment skipped: Stage 1 diagnostics not found.")
        except Exception as e:
            print(f"Warning: Failed to produce Sharpe alignment CSV: {e}")
    except Exception as e:
        print(f"Warning: Failed to save summary CSV: {e}")



In [14]:
"""
results manager
"""
def compute_spy_benchmark(test_start_date, align_indices: Optional[Iterable[pd.Index]] = None, period: str = 'test') -> Optional[Dict]:
    """Fetch SPY and compute passive buy-and-hold metrics for the requested period.

    Args:
        test_start_date: Timestamp splitting train/test
        align_indices: Optional iterable of indices to align SPY to the common
            intersection across provided indices for apples-to-apples metrics.
        period: 'test' (>= test_start_date) or 'train' (< test_start_date)

    Returns:
        Dict with 'data' and 'metrics' keys, or None on failure
    """
    try:
        spy_loader = DataLoader('SPY', START_DATE, END_DATE)
        spy_prices = spy_loader.get_prices()
        if isinstance(spy_prices, pd.Series):
            spy_prices = spy_prices.to_frame()
    except Exception as e:
        print(f"Warning: could not load SPY benchmark ({e})")
        return None

    try:
        if period == 'train':
            spy_window = spy_prices.loc[spy_prices.index < test_start_date].copy()
            period_name = 'train'
        else:
            spy_window = spy_prices.loc[spy_prices.index >= test_start_date].copy()
            period_name = 'test'
        if spy_window.empty:
            print(f"Warning: SPY has no data in {period_name} period; benchmark not added.")
            return None

        price_col = spy_window.columns[0]
        # Compute log returns (same convention as strategies)
        spy_window['passive_returns'] = (spy_window[price_col] / spy_window[price_col].shift(1)).apply(
            lambda x: np.log(x) if x > 0 else 0
        )
        spy_window['passive_returns'] = spy_window['passive_returns'].fillna(0)
        
        # IMPORTANT: Align with strategy convention where positions are shifted by 1 day
        # The first test-period strategy return is 0 due to the 1-day position shift.
        # To compare apples-to-apples, set SPY's first test-period return to 0 as well.
        if not spy_window['passive_returns'].empty:
            spy_window.iloc[0, spy_window.columns.get_loc('passive_returns')] = 0.0

        # Optional alignment to strategy indices (common intersection)
        if align_indices:
            try:
                iter_indices = list(align_indices)
                if len(iter_indices) > 0:
                    common_idx = iter_indices[0]
                    for idx in iter_indices[1:]:
                        common_idx = common_idx.intersection(idx)
                    # Slice SPY to the common index intersection
                    aligned = spy_window.loc[spy_window.index.intersection(common_idx)].copy()
                    if not aligned.empty:
                        # Reset first aligned day's passive return to 0 and recompute cumsum base
                        first_idx = aligned.index[0]
                        aligned.iloc[0, aligned.columns.get_loc('passive_returns')] = 0.0
                        spy_window = aligned
            except Exception as e:
                print(f"Warning: Could not align SPY to provided indices: {e}")

        spy_window['cum_passive_returns'] = spy_window['passive_returns'].cumsum().apply(np.exp)

        spy_metrics = PerformanceMetrics.calculate_all_metrics(
            spy_window['passive_returns'],
            spy_window['cum_passive_returns']
        )

        return {
            'data': spy_window,
            'metrics': spy_metrics
        }
    except Exception as e:
        print(f"Warning computing SPY benchmark metrics: {e}")
        return None


def save_and_print_results(results: Dict) -> pd.DataFrame:
    """Build comparison DataFrame from results, print and save to CSV.
    
    Args:
        results: Dictionary mapping strategy_name -> result dict with 'metrics' key
    
    Returns:
        Comparison DataFrame sorted by Total Return (descending)
    """
    comparison_df = pd.DataFrame({name: res['metrics'] for name, res in results.items()}).T
    
    # Sort by Total Return in descending order
    comparison_df = comparison_df.sort_values('Total Return', ascending=False)

    print("\n" + "="*80)
    print("STRATEGY COMPARISON")
    print("="*80)
    print(comparison_df.to_string())

    comparison_df.to_csv('strategy_comparison.csv')
    print("\nResults saved to strategy_comparison.csv")
    return comparison_df


def save_detailed_results(results: Dict, out_dir: str = "results") -> None:
    """Save per-strategy backtest DataFrames and a summary CSV for diagnostics.
    
    Args:
        results: Dictionary mapping strategy_name -> result dict
        out_dir: Output directory for results
    
    Writes:
        - results/strategy_data/<strategy>.csv for each strategy
        - results/summary.csv with aggregated metrics
    """
    out_path = Path(out_dir)
    data_dir = out_path / "strategy_data"
    out_path.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)

    def safe_name(name: str) -> str:
        """Convert strategy name to safe filename."""
        return re.sub(r"[^A-Za-z0-9._\-]+", "_", name)[:200]

    summary_rows = []
    for name, res in results.items():
        bt_data = res.get('data')
        final_value = res.get('final_value')
        total_return = res.get('total_return')
        metrics = res.get('metrics', {}) or {}

        # Save per-strategy data if available
        if isinstance(bt_data, pd.DataFrame) and not bt_data.empty:
            file_path = data_dir / f"{safe_name(name)}.csv"
            try:
                bt_data.to_csv(file_path, index_label='date')
            except Exception as e:
                print(f"Warning: Failed to save data for {name}: {e}")

        # Build summary row
        row = {
            'strategy': name,
            'final_value': final_value,
            'total_return_pct': total_return,
        }
        # Merge metrics (ensure flat dict)
        if isinstance(metrics, dict):
            row.update(metrics)
        summary_rows.append(row)

    # Save summary CSV
    try:
        summary_df = pd.DataFrame(summary_rows)
        summary_csv_path = out_path / 'summary.csv'
        summary_df.sort_values('total_return_pct', ascending=False).to_csv(summary_csv_path, index=False)
        print(f"Detailed results saved to: {out_path.resolve()} (data + summary.csv)")

        # Optional: Create Sharpe alignment debug CSV if Stage 1 diagnostics exist
        try:
            stage1_csv = Path(__file__).resolve().parents[1] / 'results' / 'debug_feature_engineering_sharpes.csv'
            if stage1_csv.exists():
                sharpe_df = pd.read_csv(stage1_csv)
                # Choose best feature per strategy by abs Sharpe
                idx_best = sharpe_df.groupby('strategy')['sharpe_stage1_abs'].idxmax()
                best_view = sharpe_df.loc[idx_best].reset_index(drop=True).rename(
                    columns={'feature': 'best_feature'}
                )

                # Merge with summary Sharpe Ratio as fallback if train/test comparison isn't available
                res_df = summary_df.copy()
                if 'strategy' not in res_df.columns:
                    # Attempt to find strategy column
                    if res_df.columns[0].lower() in ('strategy', 'name', 'strategy_name'):
                        res_df = res_df.rename(columns={res_df.columns[0]: 'strategy'})
                keep_cols = ['strategy'] + [c for c in res_df.columns if 'sharpe' in c.lower()]
                res_sel = res_df[keep_cols]
                merged = best_view.merge(res_sel, on='strategy', how='left')
                out_path_align = out_path / 'debug_sharpe_alignment.csv'
                merged.to_csv(out_path_align, index=False)
                print(f"Sharpe alignment (with summary.csv) saved to: {out_path_align}")
            else:
                print("Sharpe alignment skipped: Stage 1 diagnostics not found.")
        except Exception as e:
            print(f"Warning: Failed to produce Sharpe alignment CSV: {e}")
    except Exception as e:
        print(f"Warning: Failed to save summary CSV: {e}")


__all__ = ["compute_spy_benchmark", "save_and_print_results", "save_detailed_results"]


In [15]:
"""
Runner functions
"""


def load_prices_and_test_start(ticker: str = TICKER):
    """Load price data for ticker and compute time-based test start date.

    Returns: prices (DataFrame), test_start_date (Timestamp)
    """
    print(f"Loading {ticker} data from {START_DATE} to {END_DATE}...")
    loader = DataLoader(ticker, START_DATE, END_DATE)
    prices = loader.get_prices()
    if isinstance(prices, pd.Series):
        prices = prices.to_frame()

    train_pct = float(ML_PARAMS.get('train_test_split', 0.75))
    if not (0.0 < train_pct < 1.0):
        train_pct = 0.75
    split_idx = int(len(prices) * train_pct)
    split_idx = max(1, min(len(prices) - 1, split_idx))
    test_start_date = prices.index[split_idx]
    return prices, test_start_date


def run_backtests_on_test_period(strategies, test_start_date):
    """Run backtests for each strategy restricted to the test period. Returns results dict."""
    results = {}
    backtest_engine = BacktestEngine(INITIAL_CAPITAL, COMMISSION_PER_TRADE)
    
    # Print test period information
    print("\n" + "="*70)
    print("TEST PERIOD INFORMATION")
    print("="*70)
    print(f"Test Start Date: {test_start_date.date()}")
    print(f"Test End Date:   {END_DATE.date()}")
    print(f"Duration:        {(END_DATE - test_start_date).days} days")
    print("="*70)

    for strategy in strategies:
        # print(f"\nBacktesting {strategy.name}...")
        strategy.calculate_returns()

        test_mask = strategy.data.index >= test_start_date
        test_data = strategy.data.loc[test_mask].copy()

        if test_data.empty:
            print(f"Warning: no test-period rows for strategy {strategy.name}; skipping.")
            continue

        # Enforce day-1 alignment: zero-out returns on the first test-period row
        first_idx = test_data.index[0]
        if 'passive_returns' in test_data.columns:
            test_data.loc[first_idx, 'passive_returns'] = 0.0
        if 'strategy_returns' in test_data.columns:
            test_data.loc[first_idx, 'strategy_returns'] = 0.0

        # Recalculate cumulative returns for test period only (starting from 1.0)
        test_data['cum_passive_returns'] = test_data['passive_returns'].cumsum().apply(np.exp)
        test_data['cum_strategy_returns'] = test_data['strategy_returns'].cumsum().apply(np.exp)

        bt_data, final_value, total_return = backtest_engine.run_backtest(test_data)

        # Check if this is an ML strategy with MSE metrics
        train_mse = getattr(strategy, 'train_mse', None)
        test_mse = getattr(strategy, 'test_mse', None)

        metrics = PerformanceMetrics.calculate_all_metrics(
            bt_data['strategy_returns'],
            bt_data['cum_strategy_returns'],
            train_mse=train_mse,
            test_mse=test_mse,
            final_value=final_value
        )

        results[strategy.name] = {
            'data': bt_data,
            'final_value': final_value,
            'total_return': total_return,
            'metrics': metrics
        }

    return results


def run_backtests_on_train_period(strategies, test_start_date):
    """Run backtests for each strategy restricted to the train period. Returns results dict."""
    results = {}
    backtest_engine = BacktestEngine(INITIAL_CAPITAL, COMMISSION_PER_TRADE)

    # Print train period information
    print("\n" + "="*70)
    print("TRAIN PERIOD INFORMATION")
    print("="*70)
    print(f"Train Start Date: {START_DATE.date()}")
    print(f"Train End Date:   {test_start_date.date()}")
    print(f"Duration:        {(test_start_date - START_DATE).days} days")
    print("="*70)

    for strategy in strategies:
        # print(f"\nBacktesting (Train) {strategy.name}...")
        strategy.calculate_returns()

        train_mask = strategy.data.index < test_start_date
        train_data = strategy.data.loc[train_mask].copy()

        if train_data.empty:
            print(f"Warning: no train-period rows for strategy {strategy.name}; skipping.")
            continue

        # Enforce day-1 alignment: zero-out returns on the first train-period row
        first_idx = train_data.index[0]
        if 'passive_returns' in train_data.columns:
            train_data.loc[first_idx, 'passive_returns'] = 0.0
        if 'strategy_returns' in train_data.columns:
            train_data.loc[first_idx, 'strategy_returns'] = 0.0

        # Recalculate cumulative returns for train period only (starting from 1.0)
        train_data['cum_passive_returns'] = train_data['passive_returns'].cumsum().apply(np.exp)
        train_data['cum_strategy_returns'] = train_data['strategy_returns'].cumsum().apply(np.exp)

        bt_data, final_value, total_return = backtest_engine.run_backtest(train_data)

        # Check if this is an ML strategy with MSE metrics
        train_mse = getattr(strategy, 'train_mse', None)
        test_mse = getattr(strategy, 'test_mse', None)

        metrics = PerformanceMetrics.calculate_all_metrics(
            bt_data['strategy_returns'],
            bt_data['cum_strategy_returns'],
            train_mse=train_mse,
            test_mse=test_mse,
            final_value=final_value
        )

        results[strategy.name] = {
            'data': bt_data,
            'final_value': final_value,
            'total_return': total_return,
            'metrics': metrics
        }

    return results




In [16]:
"""
plotting functions
"""
def plot_price_with_signals(df: pd.DataFrame, price_col: str = None, sig_col: str = 'trades'):
    """Plot price and mark buy/sell signals (where `sig_col` != 0)."""
    if price_col is None:
        price_col = df.columns[0]

    # Prepare data
    plot_df = df[[price_col, sig_col]].copy()
    plot_df['date'] = plot_df.index
    
    trades = df[df[sig_col] != 0]
    buys = trades[trades[sig_col] > 0]
    sells = trades[trades[sig_col] < 0]
    
    buy_df = pd.DataFrame({
        'date': buys.index,
        'price': df.loc[buys.index, price_col],
        'type': 'Buy'
    })
    
    sell_df = pd.DataFrame({
        'date': sells.index,
        'price': df.loc[sells.index, price_col],
        'type': 'Sell'
    })
    
    signals_df = pd.concat([buy_df, sell_df])
    
    # Create plot with responsive size
    p = (ggplot() + 
        geom_line(aes(x='date', y=price_col), data=plot_df, color='gray', size=1) + 
        geom_point(aes(x='date', y='price', color='type', shape='type'), 
                   data=signals_df, size=3) + 
        scale_color_manual(values={'Buy': 'green', 'Sell': 'red'}) + 
        scale_shape_manual(values={'Buy': 24, 'Sell': 25}) + 
        labs(title='Price with Buy/Sell Signals',
             x='Date',
             y='Price') + 
        theme_light())
    
    return p


def plot_cumulative_returns(df: pd.DataFrame, cum_col: str = 'cum_strategy_returns'):
    """Plot cumulative returns."""
    plot_df = df[[cum_col]].copy()
    plot_df['date'] = plot_df.index
    
    p = (ggplot(plot_df, aes(x='date', y=cum_col)) + 
        geom_line(size=1.5, color='blue') + 
        labs(title='Cumulative Strategy Returns',
             x='Date',
             y='Growth Factor') + 
        theme_light())
    
    return p


def plot_top_strategy_vs_benchmark(results, comparison_df, initial_capital=100000):
    """Plot wealth curve of top strategy vs SPY with buy/sell signals.
    
    Args:
        results: Dict of strategy results from run_backtests_on_test_period
        comparison_df: DataFrame with strategy metrics sorted by Total Return
        initial_capital: Starting capital for wealth calculations
    
    Returns:
        lets_plot GGBunch with two plots
    """
    # Get top strategy name (first row, excluding SPY if present)
    top_strategies = comparison_df.index.tolist()
    top_strategy_name = None
    
    for strategy_name in top_strategies:
        if strategy_name != 'SPY':
            top_strategy_name = strategy_name
            break
    
    if top_strategy_name is None:
        print("Warning: No strategy found to plot.")
        return None
    
    # Get data for top strategy and SPY
    if top_strategy_name not in results or 'SPY' not in results:
        print(f"Warning: Missing data for {top_strategy_name} or SPY")
        return None
    
    top_data = results[top_strategy_name]['data']
    spy_data = results['SPY']['data']
    
    # Calculate wealth curves (normalized to start at initial_capital)
    top_wealth = top_data['cum_strategy_returns'] * initial_capital
    spy_wealth = spy_data['cum_passive_returns'] * initial_capital
    
    # Align the data on the same index (use inner join to ensure same length)
    wealth_df = pd.DataFrame({
        'strategy_wealth': top_wealth,
        'spy_wealth': spy_wealth
    }).dropna()  # Remove any NaN values
    
    # Reset index to get date as a column (robustly name the first column 'date')
    wealth_df = wealth_df.reset_index()
    if wealth_df.columns[0] != 'date':
        wealth_df = wealth_df.rename(columns={wealth_df.columns[0]: 'date'})
    
    # Reshape for lets-plot
    wealth_long = pd.concat([
        pd.DataFrame({
            'date': wealth_df['date'],
            'wealth': wealth_df['strategy_wealth'],
            'type': top_strategy_name
        }),
        pd.DataFrame({
            'date': wealth_df['date'],
            'wealth': wealth_df['spy_wealth'],
            'type': 'SPY (Buy & Hold)'
        })
    ])
    
    positions = top_data['positions']
    wealth_long["date"] = pd.to_datetime(wealth_long["date"])
    
    # Plot 1: Wealth curves with buy/sell signals
    p1 = (ggplot(wealth_long, aes(x='date', y='wealth', color='type')) + 
        geom_line(size=1.5, tooltips=layer_tooltips().format('@date', '%Y-%m-%d')) + 
        scale_color_manual(values={top_strategy_name: 'blue', 'SPY (Buy & Hold)': 'gray'}) + 
        labs(title=f'Wealth Curve: {top_strategy_name} vs SPY Benchmark',
             x='Date',
             y='Portfolio Value ($)',
             color='Strategy') + 
        theme_light() + 
        theme(legend_position="top", 
              panel_grid_minor='blank',
              panel_grid_major_x='blank') +
        scale_x_datetime(format="%b %Y") +
        ggsize(2000, 1200))
    
    # Add buy/sell signals if positions available
    if positions is not None:
        # Get SPY price from the data
        spy_price_col = spy_data.columns[0]
        spy_price = spy_data[spy_price_col]
        spy_price_normalized = (spy_price / spy_price.iloc[0]) * initial_capital

        # Find buy/sell signals (position state changes)
        buy_signals = (positions == 1) & (positions.shift(1) != 1)
        sell_signals = (positions != 1) & (positions.shift(1) == 1)

        signals_list = []
        if buy_signals.any():
            buy_dates = buy_signals[buy_signals].index
            buy_dates = spy_price_normalized.index.intersection(buy_dates)
            buy_prices = spy_price_normalized.loc[buy_dates]
            for date, price in zip(buy_dates, buy_prices):
                signals_list.append({'date': date, 'price': price, 'signal': 'Buy'})

        if sell_signals.any():
            sell_dates = sell_signals[sell_signals].index
            sell_dates = spy_price_normalized.index.intersection(sell_dates)
            sell_prices = spy_price_normalized.loc[sell_dates]
            for date, price in zip(sell_dates, sell_prices):
                signals_list.append({'date': date, 'price': price, 'signal': 'Sell'})

        if signals_list:
            signals_df = pd.DataFrame(signals_list)
            p1 = p1 + geom_point(aes(x='date', y='price', shape='signal', fill='signal'),
                                 data=signals_df, size=3, color='black') + \
                scale_shape_manual(values={'Buy': 24, 'Sell': 25}) + \
                scale_fill_manual(values={'Buy': 'green', 'Sell': 'red'})
    
    # Get metrics for annotation
    top_return = comparison_df.loc[top_strategy_name, 'Total Return']
    top_sharpe = comparison_df.loc[top_strategy_name, 'Sharpe Ratio']
    spy_return = comparison_df.loc['SPY', 'Total Return']
    spy_sharpe = comparison_df.loc['SPY', 'Sharpe Ratio']
    
    metrics_text = f'{top_strategy_name}: Total Return={top_return:.2%}, Sharpe={top_sharpe:.4f}\n'
    metrics_text += f'SPY: Total Return={spy_return:.2%}, Sharpe={spy_sharpe:.4f}\n'
    metrics_text += f'Alpha: {top_return - spy_return:.2%}'
    
    
    print(f"\n{metrics_text}")
    
    return p1
    

def plot_top_n_strategies_vs_benchmark(results, comparison_df, top_n=5, initial_capital=100000):
    """Plot wealth curves of top N strategies vs SPY on the same axis.
    
    Args:
        results: Dict of strategy results from run_backtests_on_test_period
        comparison_df: DataFrame with strategy metrics sorted by Total Return
        top_n: Number of top strategies to plot
        initial_capital: Starting capital for wealth calculations
    
    Returns:
        lets_plot figure
    """
    # Get top N strategy names (excluding SPY)
    top_strategies = [name for name in comparison_df.index.tolist() 
                      if name != 'SPY'][:top_n]
    
    if not top_strategies:
        print("Warning: No strategies found to plot.")
        return None
    
    # Check if SPY exists
    if 'SPY' not in results:
        print("Warning: SPY benchmark not found in results.")
        return None
    
    # Prepare data for all strategies
    wealth_data_list = []
    
    # Add SPY benchmark
    spy_data = results['SPY']['data']
    spy_wealth = spy_data['cum_passive_returns'] * initial_capital
    
    for date, wealth in zip(spy_wealth.index, spy_wealth.values):
        wealth_data_list.append({
            'date': date,
            'wealth': wealth,
            'strategy': 'SPY (Benchmark)',
            'type': 'Benchmark'
        })
    
    # Add top N strategies
    for strategy_name in top_strategies:
        if strategy_name not in results:
            continue
        
        strategy_data = results[strategy_name]['data']
        strategy_wealth = strategy_data['cum_strategy_returns'] * initial_capital
        
        for date, wealth in zip(strategy_wealth.index, strategy_wealth.values):
            wealth_data_list.append({
                'date': date,
                'wealth': wealth,
                'strategy': strategy_name,
                'type': 'Strategy'
            })
    
    wealth_df = pd.DataFrame(wealth_data_list)
    
    # Create color palette
    n_strategies = len(top_strategies)
    strategy_colors = {}
    strategy_colors['SPY (Benchmark)'] = 'gray'
    
    # Use a color palette for strategies
    color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
                     '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    
    for i, strategy_name in enumerate(top_strategies):
        strategy_colors[strategy_name] = color_palette[i % len(color_palette)]
    
    # Create plot
    p = (ggplot(wealth_df, aes(x='date', y='wealth', color='strategy')) + 
        geom_line(size=1.5, alpha=0.8) + 
        scale_color_manual(values=strategy_colors) + 
        labs(title=f'Top {len(top_strategies)} Strategies vs SPY Benchmark',
             x='Date',
             y='Portfolio Value ($)',
             color='Strategy') + 
        theme_light() + 
        theme(legend_position='right') +
        ggsize(2000, 1200))
    
    # Add performance summary
    print("\n" + "="*80)
    print(f"TOP {len(top_strategies)} STRATEGIES PERFORMANCE SUMMARY")
    print("="*80)
    
    print(f"\n{'Rank':<6} {'Strategy':<40} {'Total Return':<12} {'Sharpe':<10} {'Max DD':<10}")
    print("-" * 80)
    
    spy_return = comparison_df.loc['SPY', 'Total Return']
    spy_sharpe = comparison_df.loc['SPY', 'Sharpe Ratio']
    spy_dd = comparison_df.loc['SPY', 'Max Drawdown']
    
    print(f"{'   -':<6} {'SPY (Benchmark)':<40} {spy_return:>10.2%} {spy_sharpe:>9.4f} {-spy_dd:>9.2%}")
    print("-" * 80)
    
    for i, strategy_name in enumerate(top_strategies, 1):
        ret = comparison_df.loc[strategy_name, 'Total Return']
        sharpe = comparison_df.loc[strategy_name, 'Sharpe Ratio']
        dd = comparison_df.loc[strategy_name, 'Max Drawdown']
        alpha = ret - spy_return
        
        print(f"{i:<6} {strategy_name:<40} {ret:>10.2%} {sharpe:>9.4f} {-dd:>9.2%}")
        print(f"{'':6} {'  → Alpha vs SPY:':<40} {alpha:>10.2%}")
    
    print("="*80 + "\n")
    
    return p


# Constants and Configs

In [17]:
# Environment & path setup
from pathlib import Path
import os, sys, platform

# Ensure working directory is the notebook's folder (take_home_assignment)
nb_dir = Path(os.getcwd()).resolve()
if (nb_dir / 'utils').exists():
    pass  # already correct
else:
    # If opened from workspace root, move into take_home_assignment
    candidate = Path(__file__).parent if '__file__' in globals() else Path.cwd() / 'take_home_assignment'
    if (candidate / 'utils').exists():
        os.chdir(candidate)
        nb_dir = candidate.resolve()

# Add current directory to Python path for package imports
if str(nb_dir) not in sys.path:
    sys.path.insert(0, str(nb_dir))

print(f'Python: {platform.python_version()}')
print(f'Working dir: {Path.cwd()}')
print(f'On path: {sys.path[0]}')

Python: 3.12.0
Working dir: /Users/ju/Projects/qf627/take_home_assignment
On path: /Users/ju/Projects/qf627/take_home_assignment


In [18]:



# Initialize lets-plot (suppress HTML output in console)
LetsPlot.setup_html()

# Utility to save plots (replicated from main.py)
from pathlib import Path
def save_plot(plot_obj, filename, script_dir: Path):
    if plot_obj is None:
        return None
    plot_dir = script_dir / 'results' / 'plots'
    plot_dir.mkdir(parents=True, exist_ok=True)
    plot_path = plot_dir / filename
    ggsave(plot_obj, str(plot_path.resolve()))
    return plot_path.resolve()

script_dir = Path.cwd()
script_dir

PosixPath('/Users/ju/Projects/qf627/take_home_assignment')

## 1) Load Prices and Determine Test Start

In [19]:
prices, test_start_date = load_prices_and_test_start()
print('Test start date:', test_start_date)
print('Prices shape:', getattr(prices, 'shape', None))
display(prices.head(3))

Loading SPY data from 2006-11-01 00:00:00 to 2025-11-12 00:00:00...
Test start date: 2021-02-08 00:00:00
Prices shape: (4787, 1)


Ticker,SPY
Date,
2006-11-01,95.973579
2006-11-02,95.917488
2006-11-03,95.749214


## 2) Initialize Strategies (Feature Engineering Encapsulated)

In [20]:
strategies = init_strategies(prices, test_start_date)
print('Number of strategies:', len(strategies))
print('Sample strategies:', list(strategies)[:8])


✓ Starting fresh training run (checkpoints disabled)

Building ML features from rule-based strategies...
Built ML feature set with 194 features from 100 strategies.
Feature names: ['SMA_20_60__spread_zscore', 'SMA_50_200__spread_zscore', 'SMA_24_58__spread_zscore', 'MACD_12_26_9__macd', 'MACD_12_26_9__signal_line', 'MACD_5_35_5__macd', 'MACD_5_35_5__signal_line', 'MACD_19_39_9__macd', 'MACD_19_39_9__signal_line', 'ZScore_42__zscore', 'RSI_p14_os30_ob70__rsi', 'RSI_p14_os25_ob75__rsi', 'RSI_p21_os30_ob70__rsi', 'RSI_p21_os25_ob75__rsi', 'BB_w20_std1.5__bb_middle', 'BB_w20_std1.5__bb_upper', 'BB_w20_std1.5__bb_lower', 'BB_w20_std2.0__bb_middle', 'BB_w20_std2.0__bb_upper', 'BB_w20_std2.0__bb_lower', 'BB_w20_std2.5__bb_middle', 'BB_w20_std2.5__bb_upper', 'BB_w20_std2.5__bb_lower', 'BB_w30_std1.5__bb_middle', 'BB_w30_std1.5__bb_upper', 'BB_w30_std1.5__bb_lower', 'BB_w30_std2.0__bb_middle', 'BB_w30_std2.0__bb_upper', 'BB_w30_std2.0__bb_lower', 'BB_w30_std2.5__bb_middle', 'BB_w30_std2.5__bb_

## 3) Run Backtests on Test Period + Add SPY Benchmark

In [21]:
# Run test period backtests
results = run_backtests_on_test_period(strategies, test_start_date)

# Build alignment indices from strategies (exclude SPY if present)
align_indices = []
for name, res in results.items():
    df = res.get('data')
    if name != 'SPY' and df is not None and hasattr(df, 'index') and len(df.index) > 0:
        align_indices.append(df.index)

# Compute SPY benchmark for test period
spy_res = compute_spy_benchmark(test_start_date, align_indices=align_indices, period='test')
if spy_res is not None:
    results['SPY'] = {
        'data': spy_res['data'],
        'final_value': None,
        'total_return': None,
        'metrics': spy_res['metrics']
    }
print('Results strategies:', len(results))
sorted(list(results.keys()))[:10]


TEST PERIOD INFORMATION
Test Start Date: 2021-02-08
Test End Date:   2025-11-12
Duration:        1738 days
Results strategies: 111


['AdaBoost_34f',
 'BB_w20_std1.5',
 'BB_w20_std2.0',
 'BB_w20_std2.5',
 'BB_w30_std1.5',
 'BB_w30_std2.0',
 'BB_w30_std2.5',
 'BB_w50_std1.5',
 'BB_w50_std2.0',
 'BB_w50_std2.5']

## 4) Train-Period Backtests + SPY Benchmark

In [22]:

train_results = run_backtests_on_train_period(strategies, test_start_date)

align_indices_train = []
for name, res in train_results.items():
    df = res.get('data')
    if name != 'SPY' and df is not None and hasattr(df, 'index') and len(df.index) > 0:
        align_indices_train.append(df.index)

spy_train = compute_spy_benchmark(test_start_date, align_indices=align_indices_train, period='train')
if spy_train is not None:
    train_results['SPY'] = {
        'data': spy_train['data'],
        'final_value': None,
        'total_return': None,
        'metrics': spy_train['metrics']
    }
print('Train results strategies:', len(train_results))
sorted(list(train_results.keys()))[:10]


TRAIN PERIOD INFORMATION
Train Start Date: 2006-11-01
Train End Date:   2021-02-08
Duration:        5213 days
Train results strategies: 111


['AdaBoost_34f',
 'BB_w20_std1.5',
 'BB_w20_std2.0',
 'BB_w20_std2.5',
 'BB_w30_std1.5',
 'BB_w30_std2.0',
 'BB_w30_std2.5',
 'BB_w50_std1.5',
 'BB_w50_std2.0',
 'BB_w50_std2.5']

## 5) Save and Display Comparison Tables

In [24]:
import re

In [25]:
# Save results and obtain test comparison DataFrame
comparison_df = save_and_print_results(results)
save_detailed_results(results)

# Build train metrics table
comparison_train_df = pd.DataFrame({name: res['metrics'] for name, res in train_results.items()}).T
comparison_train_df = comparison_train_df.sort_values('Total Return', ascending=False)

# Merge Train vs Test metrics and save
merged = comparison_train_df.add_suffix(' (Train)').join(comparison_df.add_suffix(' (Test)'), how='outer')
merged_path = (Path.cwd() / 'results' / 'train_vs_test_comparison.csv')
merged.to_csv(merged_path)

print('='*70)
print('TRAIN vs TEST METRICS (saved to results/train_vs_test_comparison.csv)')
print('='*70)
display(merged.sort_values('Sharpe Ratio (Train)', ascending=False).head(12))

print('Top by Total Return (Test):')
display(comparison_df.sort_values('Total Return', ascending=False).head(10))

print('Top by Sharpe (Test):')
display(comparison_df.sort_values('Sharpe Ratio', ascending=False).head(10))


STRATEGY COMPARISON
                                                                                                   Sharpe Ratio      CAGR  Max Drawdown  Longest DD Duration  Total Return  Volatility    Final Value  Train MSE  Test MSE
Coincident_TLT_5                                                                                       1.045070  0.196809     -0.212375                333.0      1.349954    0.172116  234995.401315        NaN       NaN
XGB_34f                                                                                                0.933538  0.174163     -0.187552                526.0      1.145873    0.172191  214587.302952   0.000168  0.000117
ExtraTrees_34f                                                                                         0.901202  0.167673     -0.223139                422.0      1.090044    0.172211  209004.399970   0.000170  0.000118
LASSO_34f                                                                                              

,Sharpe Ratio (Train),CAGR (Train),Max Drawdown (Train),Longest DD Duration (Train),Total Return (Train),Volatility (Train),Final Value (Train),Train MSE (Train),Test MSE (Train),Sharpe Ratio (Test),CAGR (Test),Max Drawdown (Test),Longest DD Duration (Test),Total Return (Test),Volatility (Test),Final Value (Test),Train MSE (Test),Test MSE (Test)
KNN_34f,3.438645,1.021181,-0.234393,270.0,11231.375289,0.204823,1.123238e+09,0.000153,0.000127,-0.601850,-0.098427,-0.498705,1425.0,-0.389058,0.172364,61094.192138,0.000153,0.000127
XGB_34f,1.177076,0.278630,-0.282006,494.0,24.988957,0.209001,2.598896e+06,0.000168,0.000117,0.933538,0.174163,-0.187552,526.0,1.145873,0.172191,214587.302952,0.000168,0.000117
LinReg_34f,0.995039,0.231139,-0.325504,2861.0,14.737065,0.209164,1.573707e+06,0.000172,0.000129,-0.270719,-0.045563,-0.306264,1258.0,-0.198903,0.172463,80109.710788,0.000172,0.000129
ExtraTrees_34f,0.971466,0.225112,-0.366410,332.0,13.746035,0.209183,1.474603e+06,0.000170,0.000118,0.901202,0.167673,-0.223139,422.0,1.090044,0.172211,209004.399970,0.000170,0.000118
GBT_34f,0.960817,0.222398,-0.272828,310.0,13.318957,0.209191,1.431896e+06,0.000164,0.000118,0.763625,0.140433,-0.244964,709.0,0.868116,0.172289,186811.573722,0.000164,0.000118
RF_34f,0.890523,0.204628,-0.247487,360.0,10.792882,0.209245,1.179288e+06,0.000165,0.000117,0.841677,0.155812,-0.253523,722.0,0.990995,0.172246,199099.463054,0.000165,0.000117
ElasticNet_34f,0.846863,0.193715,-0.224674,518.0,9.452850,0.209277,1.045285e+06,0.000174,0.000118,0.346504,0.061500,-0.217443,937.0,0.328207,0.172447,132820.686595,0.000174,0.000118
SPY,0.824017,0.159678,-0.337173,280.0,4.899496,0.179910,NaN,NaN,NaN,0.763625,0.140433,-0.244964,709.0,0.868116,0.172289,NaN,NaN,NaN
LASSO_34f,0.822136,0.187576,-0.228773,600.0,8.762369,0.209294,9.762369e+05,0.000174,0.000118,0.899989,0.167430,-0.197818,543.0,1.087978,0.172212,208797.767828,0.000174,0.000118
AdaBoost_34f,0.794538,0.180759,-0.345222,372.0,8.045272,0.209312,9.045272e+05,0.000162,0.000115,0.763625,0.140433,-0.244964,709.0,0.868116,0.172289,186811.573722,0.000162,0.000115


Top by Total Return (Test):


,Sharpe Ratio,CAGR,Max Drawdown,Longest DD Duration,Total Return,Volatility,Final Value,Train MSE,Test MSE
Coincident_TLT_5,1.045070,0.196809,-0.212375,333.0,1.349954,0.172116,234995.401315,NaN,NaN
XGB_34f,0.933538,0.174163,-0.187552,526.0,1.145873,0.172191,214587.302952,0.000168,0.000117
ExtraTrees_34f,0.901202,0.167673,-0.223139,422.0,1.090044,0.172211,209004.399970,0.000170,0.000118
LASSO_34f,0.899989,0.167430,-0.197818,543.0,1.087978,0.172212,208797.767828,0.000174,0.000118
Coincident_XLC_252,0.890299,0.165492,-0.187552,478.0,1.071544,0.172218,207154.406326,NaN,NaN
RF_34f,0.841677,0.155812,-0.253523,722.0,0.990995,0.172246,199099.463054,0.000165,0.000117
Coincident_GLD_252,0.819985,0.151519,-0.212449,365.0,0.956065,0.172259,195606.528124,NaN,NaN
Coincident_EURUSD=X_252,0.813872,0.150311,-0.170491,464.0,0.946331,0.172262,194633.100861,NaN,NaN
DTree_34f,0.763625,0.140433,-0.244964,709.0,0.868116,0.172289,186811.573722,0.000169,0.000118
GBT_34f,0.763625,0.140433,-0.244964,709.0,0.868116,0.172289,186811.573722,0.000164,0.000118


Top by Sharpe (Test):


,Sharpe Ratio,CAGR,Max Drawdown,Longest DD Duration,Total Return,Volatility,Final Value,Train MSE,Test MSE
Coincident_TLT_5,1.045070,0.196809,-0.212375,333.0,1.349954,0.172116,234995.401315,NaN,NaN
XGB_34f,0.933538,0.174163,-0.187552,526.0,1.145873,0.172191,214587.302952,0.000168,0.000117
ExtraTrees_34f,0.901202,0.167673,-0.223139,422.0,1.090044,0.172211,209004.399970,0.000170,0.000118
LASSO_34f,0.899989,0.167430,-0.197818,543.0,1.087978,0.172212,208797.767828,0.000174,0.000118
Coincident_XLC_252,0.890299,0.165492,-0.187552,478.0,1.071544,0.172218,207154.406326,NaN,NaN
RF_34f,0.841677,0.155812,-0.253523,722.0,0.990995,0.172246,199099.463054,0.000165,0.000117
Coincident_GLD_252,0.819985,0.151519,-0.212449,365.0,0.956065,0.172259,195606.528124,NaN,NaN
Coincident_EURUSD=X_252,0.813872,0.150311,-0.170491,464.0,0.946331,0.172262,194633.100861,NaN,NaN
DTree_34f,0.763625,0.140433,-0.244964,709.0,0.868116,0.172289,186811.573722,0.000169,0.000118
GBT_34f,0.763625,0.140433,-0.244964,709.0,0.868116,0.172289,186811.573722,0.000164,0.000118


## 6) Generate and Save Plots

In [26]:
print('\n' + '='*70)
print('GENERATING PLOTS')
print('='*70)

# Plot 1: Top strategy vs SPY
print('Generating: Top Strategy vs SPY Benchmark...')
plot1 = plot_top_strategy_vs_benchmark(results, comparison_df)
plot1_path = save_plot(plot1, 'top_strategy_vs_spy.html', Path.cwd())
print(f'\u2713 Saved to: {plot1_path}') if plot1_path else None

# Plot 2: Top 5 strategies vs SPY
print('Generating: Top 5 Strategies vs SPY Benchmark...')
plot2 = plot_top_n_strategies_vs_benchmark(results, comparison_df, top_n=5)
plot2_path = save_plot(plot2, 'top_5_strategies_vs_spy.html', Path.cwd())
print(f'\u2713 Saved to: {plot2_path}') if plot2_path else None

print('\n' + '='*70)
print('ALL STEPS COMPLETED SUCCESSFULLY')
print('='*70 + '\n')


GENERATING PLOTS
Generating: Top Strategy vs SPY Benchmark...

Coincident_TLT_5: Total Return=135.00%, Sharpe=1.0451
SPY: Total Return=86.81%, Sharpe=0.7636
Alpha: 48.18%
✓ Saved to: /Users/ju/Projects/qf627/take_home_assignment/results/plots/top_strategy_vs_spy.html
Generating: Top 5 Strategies vs SPY Benchmark...

TOP 5 STRATEGIES PERFORMANCE SUMMARY

Rank   Strategy                                 Total Return Sharpe     Max DD    
--------------------------------------------------------------------------------
   -   SPY (Benchmark)                              86.81%    0.7636    24.50%
--------------------------------------------------------------------------------
1      Coincident_TLT_5                            135.00%    1.0451    21.24%
         → Alpha vs SPY:                            48.18%
2      XGB_34f                                     114.59%    0.9335    18.76%
         → Alpha vs SPY:                            27.78%
3      ExtraTrees_34f                      

In [28]:
plot1

In [29]:
plot2

## 7) Quick Executive Snapshot (Top 3)

In [27]:
# Derive top 3 from test comparison and print key metrics
comp = comparison_df.copy()
top3 = comp.sort_values(['Sharpe Ratio', 'Total Return'], ascending=False).head(3)
display(top3[['Sharpe Ratio', 'CAGR', 'Max Drawdown', 'Total Return']])

# Save a compact CSV of top 3
out = top3[['Sharpe Ratio', 'CAGR', 'Max Drawdown', 'Total Return']]
out_path = Path('results')/ 'top3_summary.csv'
out.to_csv(out_path)
print(f'Saved Top-3 summary to: {out_path.resolve()}')

,Sharpe Ratio,CAGR,Max Drawdown,Total Return
Coincident_TLT_5,1.045070,0.196809,-0.212375,1.349954
XGB_34f,0.933538,0.174163,-0.187552,1.145873
ExtraTrees_34f,0.901202,0.167673,-0.223139,1.090044


Saved Top-3 summary to: /Users/ju/Projects/qf627/take_home_assignment/results/top3_summary.csv
